In [1]:
import warnings
warnings.filterwarnings('ignore')

# 벡터 저장소 검색기 활용 및 검색 성능 평가

# 라이브러리 설치

## 설치하는 라이브러리의 역할

`faiss-cpu`: 벡터 검색 엔진 라이브러리, 여러 개의 문서 조각 중에서 사용자의 질문과 의미가 가장 유사한 문서를 찾는다. 벡터스토어를 구축한다.  
`rank_bm25`: 키워드 기반 검색 알고리즘 라이브러리, 단순 키워드 일치 여부를 넘어서 단어의 희소성과 빈도를 계산해서 점수화한다.    
`kiwipiepy`: 한국어 형태소 분석기 라이브러리, 한국어 문장을 단어 단위로 쪼개고 조사를 정교하게 분리한다.  
`openpyxl`: 엑셀 파일을 읽고 쓰기 위한 라이브러리, 판다스와 함께 자주 사용된다.

In [2]:
# !pip install faiss-cpu rank_bm25 kiwipiepy openpyxl

# 환경 설정

## .env 환경 변수

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

## 기본 라이브러리

In [4]:
import os, json, re
from glob import glob
from pprint import pprint
import numpy as np
import pandas as pd

from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import JSONLoader
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 벡터저장소(VectorStore)

# Chroma 저장소 생성, 문서 관리, 문서 검색

## 벡터저장소 초기화

허깅 페이스 임베딩 모델을 사용해서 Chroma 벡터저장소 만든다.

In [7]:
# 다국어 처리가 뛰어난 BAAI/bge-m3 모델을 사용해서 허깅 페이스 임베딩 모델을 만든다.
# BAAI/bge-m3 모델은 한국어, 영어 등 여러 언어를 동시에 잘 처리하며, 긴 문장도 효과적으로 벡터화할 수 있어 RAG 시스템 구축시 선호되는 모델이다.
embeddings_model = HuggingFaceEmbeddings(model='BAAI/bge-m3')

# 비어있는 Chroma 벡터저장소 만든다. 비어있는 벡터저장소를 만들 때는 from_documents() 메소드를 사용하지 않는다.
chroma_db = Chroma(
    # 임베딩 모델을 지정한다. BAAI/bge-m3 모델을 사용해서 Chroma 벡터저장소 만들때 문자를 숫자로 바꾸는 임베딩을 한다.
    embedding_function=embeddings_model,
    collection_name='sample',
    persist_directory='./chroma_db',   
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

get() 메소드는 현재 연결된 Chroma 벡터저장소에 저장된 모든 데이터를 추출하거나, 특정 조건에 맞는 데이터를 조회할 때 사용한다.

In [6]:
# chroma_db.delete_collection()

In [8]:
chroma_db.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

## 벡터저장소 관리

Chroma 벡터저장소에는 Document 객체를 저장해야 하므로 Document를 import 한다.

In [9]:
from langchain_core.documents import Document

Chroma 벡터저장소에 저장할 데이터를 준비한다.

In [10]:
# Chroma 벡터저장소에 저장할 원본 데이터
documents = [
    '인공지능은 컴퓨터 과학의 한 분야입니다.',
    '머신러닝은 인공지능의 하위 분야입니다.',
    '딥러닝은 머신러닝의 한 종류입니다.',
    '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
    '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.',
]

# Document 객체를 생성한다. Document 객체에는 부가정보(metadata)와 본문(page_content)가 포함된다.
doc_objects = []
for index, document in enumerate(documents, start=1):
    # print(index, document)
    doc = Document(
        page_content = document,
        metadata = {'source': f'AI_textbook {index}', 'chapter': f'Chapter {index}'}
    )
    # print(doc)
    doc_objects.append(doc)
print(doc_objects)

# Chroma 벡터저장소에 저장되는 Document 객체의 고유 식별자(ID)를 생성한다.
doc_ids = [f'DOC_{i}' for i in range(1, len(doc_objects) + 1)]
print(doc_ids)

[Document(metadata={'source': 'AI_textbook 1', 'chapter': 'Chapter 1'}, page_content='인공지능은 컴퓨터 과학의 한 분야입니다.'), Document(metadata={'source': 'AI_textbook 2', 'chapter': 'Chapter 2'}, page_content='머신러닝은 인공지능의 하위 분야입니다.'), Document(metadata={'source': 'AI_textbook 3', 'chapter': 'Chapter 3'}, page_content='딥러닝은 머신러닝의 한 종류입니다.'), Document(metadata={'source': 'AI_textbook 4', 'chapter': 'Chapter 4'}, page_content='자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.'), Document(metadata={'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}, page_content='컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.')]
['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5']


Chroma 벡터저장소에 데이터를 저장한다.

In [11]:
# Chroma 벡터저장소에 데이터(Document 객체)를 추가한다.
# from_documents() 메소드는 벡터저장소를 만듬과 동시에 데이터가 저장되지만 기존 벡터저장소에 새로운 데이터를 추가하려면 add_documents() 메소드를 사용한다.
added_doc_ids = chroma_db.add_documents(
    documents=doc_objects, # 벡터저장소에 저장할 데이터를 지정한다. 저장할 데이터 타입은 Document 객체가 저장된 리스트 타입이어야 한다.
    ids=doc_ids
)

In [12]:
# add_documents() 메소드는 벡터저장소에 데이터를 추가하고 ids를 리턴한다. len() 함수를 사용해서 추가된 데이터의 개수를 얻어올 수 있다.
print(len(added_doc_ids))

5


In [13]:
chroma_db.get()

{'ids': ['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5'],
 'embeddings': None,
 'documents': ['인공지능은 컴퓨터 과학의 한 분야입니다.',
  '머신러닝은 인공지능의 하위 분야입니다.',
  '딥러닝은 머신러닝의 한 종류입니다.',
  '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
  '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'AI_textbook 1', 'chapter': 'Chapter 1'},
  {'source': 'AI_textbook 2', 'chapter': 'Chapter 2'},
  {'source': 'AI_textbook 3', 'chapter': 'Chapter 3'},
  {'chapter': 'Chapter 4', 'source': 'AI_textbook 4'},
  {'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}]}

## 유사도 검사를 이용한 문서 검색

주어진 쿼리와 가장 유사한 문서를 유사도가 높은 순서대로 지정한 개수 만큼 반환한다.

similarity_search() 메소드는 벡터저장소를 검색기로 만들지 않은 상태에서 유사도 검색을 해서 유사도가 높은 순서대로 지정한 개수 만큼 얻어온다.

In [14]:
query = '인공지능과 머신러닝의 관계는?'
# chroma 벡터저장소에서 유사도 검색을 한다.
# similarity_search() 메소드는 벡터저장소에서 유사도 검사를 한다. 검색기(retriever)는 만들지 않은 상태이다.
results = chroma_db.similarity_search(query, k=2)

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for result in results:
    print(f'- {result.page_content} [출처: {result.metadata["source"]}, {result.metadata["chapter"]}]')

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook 2, Chapter 2]
- 딥러닝은 머신러닝의 한 종류입니다. [출처: AI_textbook 3, Chapter 3]


## 문서 수정

chroma 벡터저장소의 특정 문서를 데이터를 식별하는 ID를 기준으로 내용을 덮어씌워서 수정한다.

In [15]:
# 수정할 새로운 문서 객체를 생성한다.
update_document1 = Document(
    page_content = '인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다.',
    metadata = {'source': 'AI_textbook', 'chapter': f'Chapter 11'}
)

update_document2 = Document(
    page_content = '머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다.',
    metadata = {'source': 'AI_textbook', 'chapter': f'Chapter 22'}
)

update_document3 = Document(
    page_content = '딥러닝은 머신러닝의 한 종류로, 심층 신경망을 사용하여 학습합니다.',
    metadata = {'source': 'AI_textbook', 'chapter': f'Chapter 33'}
)

In [16]:
# 단일 문서 수정
# update_document() 메소드로 수정할 ID 한 개와 수정할 내용을 지정해서 해당 데이터를 교체한다.
chroma_db.update_document(document_id='DOC_1', document=update_document1)

In [17]:
# 여러 문서 일괄 수정
# update_documents() 메소드로 수정할 ID 여러 개와 수정할 내용을 지정해서 해당 데이터를 교체한다. 리스트로 묶어서 넘겨야 한다.
chroma_db.update_documents(ids=['DOC_2', 'DOC_3'], documents=[update_document2, update_document3])

In [18]:
chroma_db.get()

{'ids': ['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5'],
 'embeddings': None,
 'documents': ['인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다.',
  '머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다.',
  '딥러닝은 머신러닝의 한 종류로, 심층 신경망을 사용하여 학습합니다.',
  '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
  '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'AI_textbook', 'chapter': 'Chapter 11'},
  {'source': 'AI_textbook', 'chapter': 'Chapter 22'},
  {'chapter': 'Chapter 33', 'source': 'AI_textbook'},
  {'source': 'AI_textbook 4', 'chapter': 'Chapter 4'},
  {'chapter': 'Chapter 5', 'source': 'AI_textbook 5'}]}

In [19]:
query = '인공지능과 머신러닝의 관계는?'
results = chroma_db.similarity_search(query, k=2)
print(f'쿼리: {query}')
print('가장 유사한 문서:')
for result in results:
    print(f'- {result.page_content} [출처: {result.metadata["source"]}, {result.metadata["chapter"]}]')

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 머신러닝은 데이터로부터 학습하여 예측과 결정을 내리는 인공지능의 하위 분야입니다. [출처: AI_textbook, Chapter 22]
- 인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다. [출처: AI_textbook, Chapter 11]


## 문서 삭제

chroma 벡터저장소의 특정 문서를 데이터를 식별하는 ID를 기준으로 삭제한다.

In [20]:
# delete() 메소드로 삭제할 ID 한 개를 지정하면 해당 ID의 문서 한 개가 삭제된다.
chroma_db.delete(ids='DOC_1')

In [21]:
# delete() 메소드로 삭제할 ID 두 개 이상을 리스트로 묶어서 지정하면 해당 ID의 문서 여러 개가 삭제된다.
chroma_db.delete(ids=['DOC_2', 'DOC_3'])

In [22]:
chroma_db.get()

{'ids': ['DOC_4', 'DOC_5'],
 'embeddings': None,
 'documents': ['자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
  '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'chapter': 'Chapter 4', 'source': 'AI_textbook 4'},
  {'chapter': 'Chapter 5', 'source': 'AI_textbook 5'}]}

In [23]:
# delete_collection() 메소드는 테이블 자체를 제거한다. 따라서, 그 안에 저장된 모든 데이터도 삭제된다.
# delete_collection() 메소드 실행 후 get() 메소드를 실행하면 delete_collection() 메소드에 의해서 내용을 확인할 테이블 자체가 삭제되기 때문에 에러가 발생된다.
chroma_db.delete_collection()

## 문서 검색

In [24]:
chroma_db = Chroma(
    embedding_function=embeddings_model,
    collection_name='sample',
    persist_directory='./chroma_db',   
)

added_doc_ids = chroma_db.add_documents(
    documents=doc_objects,
    ids=doc_ids
)

chroma_db.get()

{'ids': ['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5'],
 'embeddings': None,
 'documents': ['인공지능은 컴퓨터 과학의 한 분야입니다.',
  '머신러닝은 인공지능의 하위 분야입니다.',
  '딥러닝은 머신러닝의 한 종류입니다.',
  '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
  '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'chapter': 'Chapter 1', 'source': 'AI_textbook 1'},
  {'chapter': 'Chapter 2', 'source': 'AI_textbook 2'},
  {'chapter': 'Chapter 3', 'source': 'AI_textbook 3'},
  {'source': 'AI_textbook 4', 'chapter': 'Chapter 4'},
  {'chapter': 'Chapter 5', 'source': 'AI_textbook 5'}]}

similarity_search() 메소드에 filter 속성을 이용해서 메타데이터가 특정 조건을 만족하는 문서만 얻어올 수 있다.

In [25]:
query = '인공지능과 머신러닝의 관계는?'
results = chroma_db.similarity_search(
    query, # 질문
    k=2, # 유사도가 높은 상위 문서 개수
    # 벡터저장소에 저장된 문서들에서 메타데이터의 'source'가 'AI_textbook 3'인 문서들 중에서만 검색을 실행한다.
    filter={'source': 'AI_textbook 3'}
)
print(f'쿼리: {query}')
print('가장 유사한 문서:')
for result in results:
    print(f'- {result.page_content} [출처: {result.metadata["source"]}, {result.metadata["chapter"]}]')

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 딥러닝은 머신러닝의 한 종류입니다. [출처: AI_textbook 3, Chapter 3]


유사도 점수를 함께 얻어온다.  
유사도 점수는 거리 기준으로 점수가 산정되기 때문에 유사도 점수가 낮을수록 더 유사한 것을 의미한다.

similarity_search_with_score() 메소드는 검색 결과로 문서뿐만 아니라 질문과의 거리(유사도)를 반환한다.

In [26]:
query = '인공지능과 머신러닝의 관계는?'
# similarity_search_with_score() 메소드는 검색 결과로 Document 객체와 질문과의 거리를 튜플로 묶어서 리턴한다.
results = chroma_db.similarity_search_with_score(query, k=2)
# print(results[0])

print(f'쿼리: {query}')
print('가장 유사한 문서:')
# results에는 (문서, 거리) 형태의 튜플이 리스트 형태로 저장되어 있다.
for doc, score in results:
    print(f'- 질문과의 거리: {score:.4f}')
    print(f'- {doc.page_content} [출처: {doc.metadata["source"]}, {doc.metadata["chapter"]}]')
    print('-' * 100)

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 질문과의 거리: 0.6592
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook 2, Chapter 2]
----------------------------------------------------------------------------------------------------
- 질문과의 거리: 0.8327
- 딥러닝은 머신러닝의 한 종류입니다. [출처: AI_textbook 3, Chapter 3]
----------------------------------------------------------------------------------------------------


관련성 점수를 함께 얻어온다.  
유사도 점수는 거리 기준으로 점수가 산정되기 때문에 유사도 점수가 낮을수록 더 유사한 것을 의미하지만 관련성 점수(정확도)는 높을 수록 더 관련성이 높음을 의미한다.

similarity_search_with_relevance_scores() 메소드는 검색 결과로 문서뿐만 아니라 질문과의 관련성 점수를 반환한다.

In [27]:
query = '인공지능과 머신러닝의 관계는?'
# similarity_search_with_relevance_scores 메소드는 검색 결과로 Document 객체와 질문과의 관련성 점수를 튜플로 묶어서 리턴한다.
# 1에 가까워질수록 관련이 높음(유사함)을 의미하고 0에 가까워질수록 관련이 낮음을 의미한다.
results = chroma_db.similarity_search_with_relevance_scores(query, k=2)

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for doc, score in results:
    print(f'- 관련성 점수: {score:.4f}')
    print(f'- {doc.page_content} [출처: {doc.metadata["source"]}, {doc.metadata["chapter"]}]')
    print('-' * 100)

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 관련성 점수: 0.5339
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook 2, Chapter 2]
----------------------------------------------------------------------------------------------------
- 관련성 점수: 0.4112
- 딥러닝은 머신러닝의 한 종류입니다. [출처: AI_textbook 3, Chapter 3]
----------------------------------------------------------------------------------------------------


# FAISS(Facebook AI Similarity Search) 저장소 생성, 문서 관리, 문서 검색

FAISS는 Facebook AI의 유사도 검색 라이브러리로 효율적인 벡터 저장소 검색 및 클러스터링을 위한 오픈소스 벡터저장소이다.

주요 특징  
&nbsp;&nbsp;&nbsp;▶ 대규모 벡터 데이터셋에서 효율적인 검색이 가능하다.  
&nbsp;&nbsp;&nbsp;▶ RAM을 효율적으로 활용해서 대용량 데이터셋 처리가 가능하다.  
&nbsp;&nbsp;&nbsp;▶ GPU 가속을 지원한다.(faiss-gpu 설치가 필요하다.)  
&nbsp;&nbsp;&nbsp;▶ 다양한 인덱싱 알고리즘을 제공하여 속도와 정확도를 조절 가능하다.  

## FAISS를 사용하기 위해 필요한 라이브러리

Meta(Facebook)에서 개발한 벡터 유사도 검색 라이브러리인 FAISS를 사용하기 위해 import 한다.  

In [28]:
import faiss

복잡한 faiss 명령어를 직접 쓰지 않고 add_documents() 메소드나 similarity_search() 같은 LangChain 표준 메소드를 사용하기 위해 FAISS를 import 한다.

In [29]:
from langchain_community.vectorstores import FAISS

메모리 상에서 문서(Document) 데이터를 Key-Value(ID-문서) 형태로 저장하고 관리하는 가장 간단한 인메모리 문서 저장소를 사용하기 위해 import 한다.  
벡터와 연결된 실제 데이터(문서 내용)을 저장하는 메모리 기반 저장소(창고)이다.

In [30]:
from langchain_community.docstore.in_memory import InMemoryDocstore

## 벡터저장소 초기화

FAISS 인덱스 초기화

FAISS는 데이터를 저장하거나 검색할 때 `차원 수가 일치`해야 한다.  
따라서, 저장할 데이터가 몇 차원인지 미리 알아야 하기때문에 아래 코드를 실행해서 1024 개의 숫자로 이루어진 데이터들을 저장할 준비를 한다.

허깅 페이스의 `BAAI/bge-m3` 임베딩 모델을 사용해서 `embeddings_model.embed_query('hello world')` 명령으로 'hello world'라는 문자열을 임베딩(숫자로 변환)한 결과를 len() 함수를 실행해서 개수를 확인하면 `BAAI/bge-m3` 모델의 차원 수와 같다.  

In [31]:
# IndexFlatL2() 메소드는 L2(유클리드) 거리 방식을 사용하는 벡터들의 고차원 공간상의 위치 및 이웃 검색 알고리즘을 관리하는 인메모리 벡터 인덱스 클래스를 만든다.
faiss_index = faiss.IndexFlatL2(len(embeddings_model.embed_query('hello world')))
print('FAISS 인덱스 초기화 완료')

FAISS 인덱스 초기화 완료


`d` 속성은 'dimension'의 약자로 이 인덱스에 저장될 벡터(숫자 배열)의 길이를 얻어온다.

In [32]:
faiss_index.d

1024

FAISS 수치 인덱스를 기반으로, 텍스트 문서 원본과 벡터저장소를 결합하는 객체를 만든다.

In [33]:
# FAISS 클래스의 생성자로 임베딩 모델, 인덱스 객체, 저장소, ID와 문서를 매핑하는 사전을 넘겨서 FAISS 객체를 만든다.
faiss_db = FAISS(
    # 임베딩 모델을 지정한다. BAAI/bge-m3 모델을 사용해서 FAISS 벡터저장소 만들때 문자를 숫자로 바꾸는 임베딩을 한다.
    embedding_function=embeddings_model,
    # 실제 벡터 검색 연산을 담당하는 FAISS의 인덱스 객체를 지정한다.
    index=faiss_index,
    # 검색 결과로 반환할 텍스트 데이터와 메타데이터를 메모리상에 저장하고 관리하는 저장소를 지정한다.
    docstore=InMemoryDocstore(),
    # FAISS의 인덱스와 docstore에 저장된 문서의 고유 ID를 연결하는 사전으로 사용할 객체를 지정한다.
    # '숫자 벡터 1번이 실제 문장 A이다'라는 연결 내용을 기억할 빈 딕셔너리를 준비하고 데이터를 추가함에 따라 매핑 정보가 쌓인다.
    index_to_docstore_id={},
)

현재 FAISS 벡터저장소(faiss_db)에 실제로 저장된 데이터의 총 개수를 확인한다.  
index는 벡터저장소 내부에 실제 숫자 계산과 저장을 담당하는 FAISS 인덱스 엔진에 접근하는 속성이다.  
ntotal는 FAISS 인덱스에 등록된 전체 데이터의 개수를 기억하는 속성이다.

In [34]:
faiss_db.index.ntotal

0

## FAISS  벡터저장소에 문서 추가하기

FAISS 벡터저장소에 저장할 데이터를 준비한다.

In [35]:
# FAISS 벡터저장소에 저장할 원본 데이터
documents = [
    '인공지능은 컴퓨터 과학의 한 분야입니다.',
    '머신러닝은 인공지능의 하위 분야입니다.',
    '딥러닝은 머신러닝의 한 종류입니다.',
    '자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.',
    '컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.',
]

# Document 객체를 생성한다. Document 객체에는 부가정보(metadata)와 본문(page_content)가 포함된다.
doc_objects = []
for index, document in enumerate(documents, start=1):
    doc = Document(
        page_content = document,
        metadata = {'source': f'AI_textbook {index}', 'chapter': f'Chapter {index}'}
    )
    doc_objects.append(doc)
print(doc_objects)

# FAISS 벡터저장소에 저장되는 Document 객체의 고유 식별자(ID)를 생성한다.
doc_ids = [f'DOC_{i}' for i in range(1, len(doc_objects) + 1)]
print(doc_ids)

[Document(metadata={'source': 'AI_textbook 1', 'chapter': 'Chapter 1'}, page_content='인공지능은 컴퓨터 과학의 한 분야입니다.'), Document(metadata={'source': 'AI_textbook 2', 'chapter': 'Chapter 2'}, page_content='머신러닝은 인공지능의 하위 분야입니다.'), Document(metadata={'source': 'AI_textbook 3', 'chapter': 'Chapter 3'}, page_content='딥러닝은 머신러닝의 한 종류입니다.'), Document(metadata={'source': 'AI_textbook 4', 'chapter': 'Chapter 4'}, page_content='자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.'), Document(metadata={'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}, page_content='컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.')]
['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5']


FAISS 벡터저장소에 데이터를 저장한다.

In [36]:
# add_documents() 메소드는 FAISS 벡터저장소에 새로운 데이터(Document 객체)를 추가한다.
added_doc_ids = faiss_db.add_documents(documents=doc_objects, ids=doc_ids)
print(len(added_doc_ids))
print(added_doc_ids)

5
['DOC_1', 'DOC_2', 'DOC_3', 'DOC_4', 'DOC_5']


In [37]:
faiss_db.index.ntotal

5

## FAISS 벡터저장소에 문서 삭제하기

FAISS 벡터저장소의 특정 문서를 데이터를 식별하는 ID를 기준으로 삭제한다.

In [38]:
# delete() 메소드로 삭제할 ID 한 개를 지정하면 해당 ID의 문서 한 개가 삭제된다.
# Chroma 벡터저장소에서 delete() 메소드로 데이터 1건을 삭제할 때 []로 묶지 않아도 되지만 FAISS 벡터저장소 1건을 삭제해도 []로 묶어야 한다.
# Chroma 벡터저장소 존재하지 않는 문서의 ID를 지정해도 에러가 발생하지 않았지만 FAISS 벡터저장소 존재하지 않는 문서의 ID를 지정하면 에러가 발생된다.
faiss_db.delete(ids=['DOC_1'])

True

In [39]:
# delete() 메소드로 삭제할 ID 두 개 이상을 리스트로 묶어서 지정하면 해당 ID의 문서 여러 개가 삭제된다.
faiss_db.delete(ids=['DOC_2', 'DOC_3'])

True

reset() 메소드는 메모리에 올려둔 FAISS 객체 구조는 유지하면서 저장된 모든 벡터와 Docstore 내용을 삭제한다.

In [40]:
faiss_db.index.reset()

In [41]:
faiss_db.index.ntotal

0

## FAISS 벡터저장소에 문서 수정하기

Chroma 벡터저장소 update_document() 메소드나 update_documents() 메소드를 사용해서 직접 문서를 수정할 수 있지만 FAISS 벡터저장소는 직접적으로 문서를 수정하는 메도를 제공하지 않는다.



In [42]:
faiss_db = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

added_doc_ids = faiss_db.add_documents(documents=doc_objects, ids=doc_ids)
faiss_db.index.ntotal

5

Chroma 벡터저장소 get() 메소드를 실행하면 벡터저장소에 저장된 모든 데이터를 볼 수 있다. FAISS 벡터저장소는 get() 메소드를 제공하지 않기 때문에 데이터가 저장되는 docstore에서 `_dict` 속성으로 실제 저장된 데이터에 접근해서 values() 메소드를 실행하면 FAISS 벡터저장소에 접근된 모든 데이터를 확인할 수 있다.

In [43]:
list(faiss_db.docstore._dict.values())

[Document(id='DOC_1', metadata={'source': 'AI_textbook 1', 'chapter': 'Chapter 1'}, page_content='인공지능은 컴퓨터 과학의 한 분야입니다.'),
 Document(id='DOC_2', metadata={'source': 'AI_textbook 2', 'chapter': 'Chapter 2'}, page_content='머신러닝은 인공지능의 하위 분야입니다.'),
 Document(id='DOC_3', metadata={'source': 'AI_textbook 3', 'chapter': 'Chapter 3'}, page_content='딥러닝은 머신러닝의 한 종류입니다.'),
 Document(id='DOC_4', metadata={'source': 'AI_textbook 4', 'chapter': 'Chapter 4'}, page_content='자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.'),
 Document(id='DOC_5', metadata={'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}, page_content='컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.')]

기존 ID 삭제 후 수정된 문서 재등록 한다.

In [44]:
faiss_db.delete(ids=['DOC_1'])
update_document1 = Document(
    page_content = '인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다.',
    metadata = {'source': 'AI_textbook', 'chapter': f'Chapter 11'}
)
added_doc_ids = faiss_db.add_documents(documents=[update_document1], ids=['DOC_1'])

## FAISS 벡터저장소에 문서 검색하기

similarity_search() 메소드로 유사도로 문서 검색

In [45]:
query = '인공지능과 머신러닝의 관계는?'
# FAISS 벡터저장소에서 유사도 검색을 한다.
results = faiss_db.similarity_search(query, k=2)

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for result in results:
    print(f'- {result.page_content} [출처: {result.metadata["source"]}, {result.metadata["chapter"]}]')

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook 2, Chapter 2]
- 인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다. [출처: AI_textbook, Chapter 11]


filter를 지정한 문서 검색

In [46]:
query = '인공지능과 머신러닝의 관계는?'
results = faiss_db.similarity_search(query, k=2, filter={'source': 'AI_textbook 3'})

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for result in results:
    print(f'- {result.page_content} [출처: {result.metadata["source"]}, {result.metadata["chapter"]}]')

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 딥러닝은 머신러닝의 한 종류입니다. [출처: AI_textbook 3, Chapter 3]


similarity_search_with_score() 메소드로 문서 및 유사도 점수 검색

In [47]:
query = '인공지능과 머신러닝의 관계는?'
results = faiss_db.similarity_search_with_score(query, k=2)

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for doc, score in results:
    print(f'- 질문과의 거리: {score:.4f}')
    print(f'- {doc.page_content} [출처: {doc.metadata["source"]}, {doc.metadata["chapter"]}]')
    print('-' * 100)

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 질문과의 거리: 0.6592
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook 2, Chapter 2]
----------------------------------------------------------------------------------------------------
- 질문과의 거리: 0.6784
- 인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다. [출처: AI_textbook, Chapter 11]
----------------------------------------------------------------------------------------------------


similarity_search_with_relevance_scores() 메소드로 문서 및 관련성 점수 검색

In [48]:
query = '인공지능과 머신러닝의 관계는?'
results = faiss_db.similarity_search_with_relevance_scores(query, k=2)

print(f'쿼리: {query}')
print('가장 유사한 문서:')
for doc, score in results:
    print(f'- 관련성 점수: {score:.4f}')
    print(f'- {doc.page_content} [출처: {doc.metadata["source"]}, {doc.metadata["chapter"]}]')
    print('-' * 100)

쿼리: 인공지능과 머신러닝의 관계는?
가장 유사한 문서:
- 관련성 점수: 0.5339
- 머신러닝은 인공지능의 하위 분야입니다. [출처: AI_textbook 2, Chapter 2]
----------------------------------------------------------------------------------------------------
- 관련성 점수: 0.5203
- 인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다. [출처: AI_textbook, Chapter 11]
----------------------------------------------------------------------------------------------------


## 로컬로 저장 및 로드하기

docstore 속성에 InMemoryDocstore 객체를 이용해서 메모리 상에 임시로 존재하는 벡터저장소를 컴퓨터의 물리적인 공간에 파일로 저장한다.

컴퓨터의 물리적인 공간에 저장하면 아래와 같은 2개의 파일이 생성된다.  
`index.faiss`: 검색을 위한 벡터 데이터가 저장되는 파일  
`index.pkl`: 문서의 본문 내용과 메타데이터가 저장되는 피클 파일

In [49]:
# save_local() 메소드의 인수로 FAISS 벡터저장소의 데이터가 저장될 경로를 넘겨서 저장한다.
faiss_db.save_local('faiss_db')

In [50]:
# FAISS 벡터저장소의 index에서 reset() 메소드를 실행하면 FAISS 벡터저장소의 docstore와 연결된 인덱스가 제거된다.
faiss_db.index.reset()
faiss_db.index.ntotal

0

In [51]:
# FAISS 벡터저장소의 docstore._dict에서 clear()를 실제 저장된 본문과 메타데이터가 제거된다.
faiss_db.docstore._dict.clear()
list(faiss_db.docstore._dict.values())

[]

컴퓨터의 물리적인 공간에 파일로 저장된 FAISS 벡터저장소를 메모리로 불러온다.

In [52]:
# load_local() 메소드의 인수로 FAISS 벡터저장소가 저장된 폴더, 임베딩 모델, 피클 파일 역직렬화 허용 여부를 넘겨서 메모리로 불러온다.
faiss_db = FAISS.load_local(
    # 메모리로 불러올 파일이 저장된 폴더의 (save_local() 메소드에서 지정한)경로를 지정한다.
    folder_path='faiss_db',
    # 메모리로 불러올 FAISS 벡터저장소를 만들때 사용한 인코딩 방식을 지정한다.
    embeddings=embeddings_model,
    # FAISS 벡터저장소 저장 포맷인 피클 파일을 역직렬화하는 것을 허용한다는 의미로 True로 지정한다.
    allow_dangerous_deserialization=True
)

In [53]:
faiss_db.index.ntotal
list(faiss_db.docstore._dict.values())

[Document(id='DOC_2', metadata={'source': 'AI_textbook 2', 'chapter': 'Chapter 2'}, page_content='머신러닝은 인공지능의 하위 분야입니다.'),
 Document(id='DOC_3', metadata={'source': 'AI_textbook 3', 'chapter': 'Chapter 3'}, page_content='딥러닝은 머신러닝의 한 종류입니다.'),
 Document(id='DOC_4', metadata={'source': 'AI_textbook 4', 'chapter': 'Chapter 4'}, page_content='자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.'),
 Document(id='DOC_5', metadata={'source': 'AI_textbook 5', 'chapter': 'Chapter 5'}, page_content='컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.'),
 Document(id='DOC_1', metadata={'source': 'AI_textbook', 'chapter': 'Chapter 11'}, page_content='인공지능은 컴퓨터 과학의 핵심 분야 중 하나로, 머신러닝과 딥러닝을 포함합니다.')]

# RAG 검색기

# 의미 기반 검색(Semantic Search) - VectorStore Retriever

의미 기반 검색

텍스트를 벡터 공간에 매핑하여 의미적 유사성을 계산해서 쿼리의 문자 그대로의 의미가 아닌, 의도와 맥락을 이해하여 검색을 수행한다.

작동 원리  
&nbsp;&nbsp;&nbsp;▶ 문서 임베딩: 모든 문서를 문서 조각으로 변환해서 벡터스토어에 사전에 훈련된 언어 모델을 사용하여 임베딩을 생성해서 저장한다.  
&nbsp;&nbsp;&nbsp;▶ 쿼리 임베딩: 사용자의 검색 쿼리도 문서 임베딩과 동일한 방식으로 변환한다.  
&nbsp;&nbsp;&nbsp;▶ 유사도 계산: 코사인 유사도나 유클라디안 거리를 사용해서 쿼리 벡터와 문서 벡터 간의 유사도를 계산한다.  
&nbsp;&nbsp;&nbsp;▶ 결과 반환: 가장 유사한 문서들을 검색 결과로 반환한다.

장점  
&nbsp;&nbsp;&nbsp;▶ 동의어, 관련어 등을 고려한 더 정확한 검색 결과를 제공한다.  
&nbsp;&nbsp;&nbsp;▶ 언어의 느낌과 맥락을 이해하여 검색한다.  
&nbsp;&nbsp;&nbsp;▶ 키워드 기반 검색에서 놓칠 수 있는 정보도 검색 가능하다.

한계  
&nbsp;&nbsp;&nbsp;▶ 대규모 데이터셋에서 계산 비용이 높을 수 있다.  
&nbsp;&nbsp;&nbsp;▶ 임베딩 모델의 품질에 크게 의존한다.  
&nbsp;&nbsp;&nbsp;▶ 매우 특정한 키워드 검색에서는 전통적인 방법보다 성능이 떨어질 수 있다.

## 벡터저장소 초기화

한국어 텍스트 파일들을 불러와서 BAAI/bge-m3 모델의 토크나이저를 기준으로 검색하기 좋게 작게 문서 조각(청크)를 만드는 전처리

불러오려는 텍스트 파일의 목록을 넘겨받아 파일을 읽어서 하나로 합쳐 리턴하는 함수를 선언한다.

In [54]:
def load_txt_files(korean_txt_files):
    data = []
    for txt_file in korean_txt_files:
        loader = TextLoader(txt_file, encoding='utf-8')
        data += loader.load()
    return data

불러오려는 파일 목록을 얻어와서 하나로 합쳐주는 함수를 실행한다.

In [55]:
korean_txt_files = glob(os.path.join('./data', '*_KR.txt'))
print(korean_txt_files)
korean_data = load_txt_files(korean_txt_files)
korean_data

['./data\\리비안_KR.txt', './data\\테슬라_KR.txt']


[Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다. 2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다. 주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.\n\n리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다. 이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다. 리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다. 2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.\n\n리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.\n'),
 Document(metadata={'source': './data\\테슬라_KR.txt'}, page_content='테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. 머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.\n\n2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다. 2012년부터 2023년 

불러온 한글 문서를 문서 조각으로 만들기 위해서 토크나이저와 스플리터를 설정한다.

In [56]:
# from_pretrained() 메소드로 허깅 페이스가 학습시켜 제공하는 BAAI/bge-m3 모델에 적용된 토크나이저를 자동으로 가져와서 토크나이저를 설정한다.
tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-m3')

# from_huggingface_tokenizer() 메소드의 인수로 허깅 페이스 토크나이저, 구분자, 청크 크기, 청크간 겹치는 정도를 넘겨서 스플리터를 설정한다.
text_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    # 허깅 페이스의 'BAAI/bge-m3' 모델에 사용한 토크나이저를 토크나이저로 지정한다.
    tokenizer=tokenizer,
    # 텍스트를 나눌 구분자를 정규 표현식을 사용해서 구두점(마침표, 느낌표, 쉼표) 뒤 공백이 한 개이상 나오는 지점으로 지정한다.
    separator=r'(?<=[.!?])\s+',
    # 분할되는 문서 조각(청크)의 최대 크기를 100 토큰으로 제한한다.
    chunk_size=100,
    # 문맥 절단을 방지하기 위해 이전 문서 조각의 끝부분 일부가 다음 문서 조각의 시작 부분에 겹치는 정도를 지정한다.
    chunk_overlap=0,
    # separator에서 설정한 구분자가 단순 문자열인지 정규 표현식인지 알려준다.
    is_separator_regex=True,
    # 텍스트를 나눈 후, 분할의 기준이 되었던 구분자를 버릴지 유지할지 알려준다.
    keep_separator=False
)

스플리터로 읽어온 문서를 분할한다.

In [57]:
# split_documents()의 인수로 텍스트 파일에서 읽어와 하나로 합친 텍스트를 넘겨서 문서 조각으로 분할한다.
korean_docs = text_splitter.split_documents(korean_data)
print(len(korean_docs))
korean_docs

6


[Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.'),
 Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다.이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다.'),
 Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다.2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.'),
 Document(metadata={'source': './data\\테슬라_KR.txt'}, page_content='테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다.2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다.'),
 Document(metadata={'source': './data\\테슬라_KR.txt'}, page_cont

문서 조각으로 나눠진 한국어 문서들을 Chroma 벡터저장소로 저장한다.

In [60]:
# 토크나이저를 만들 때 'BAAI/bge-m3' 모델에서 사용한 토크나이저를 지정했으므로 임베딩 모델도 'BAAI/bge-m3' 모델로 지정한다.
embedding_huggingface = HuggingFaceEmbeddings(model='BAAI/bge-m3')

# from_documents() 메소드로 벡터저장소에 저장할 문서, 임베딩 모델, 테이블 이름, 저장될 경로를 넘겨서 Chroma 벡터저장소를 만든다.
chroma_db = Chroma.from_documents(
    # 읽어들인 한국어 문서가 문서 조각으로 분할된 벡터저장소에 저장할 문서를 지정한다.
    documents=korean_docs,
    # 벡터저장소에 문서 조각을 저장할 때 숫자로 변경할 임베딩 모델을 지정한다.
    embedding=embedding_huggingface,
    collection_name='korean_db',
    persist_directory='./chroma_db',
    # 벡터 간의 유사도 측정에 사용했던 기본값(l2, 유클리드 거리)을 의미적 유사성 판단에 더 유리한 방식인 코사인 유사도 방식을 사용한다고 설정한다.
    # 코사인 유사도는 RAG 텍스트 검색에서 가장 흔하게 사용된다.
    collection_metadata={'hnsw:space': 'cosine'}
)

len(chroma_db.get()['ids']) # Chroma 벡터저장소에 저장된 문서 조각의 개수

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

6

In [59]:
# chroma_db.delete_collection()

## 벡터검색기 초기화

검색된 문서가 실제 쿼리와 얼마나 비슷한지 유사도를 계산하기 위해서 cosine_similarity를 import 한다.

In [61]:
from langchain_community.utils.math import cosine_similarity

Chroma 벡터저장소를 벡터검색기로 만들어서 사용자 질문에 가장 적합한(유사도가 높은) 문서 조각을 얻어온다.

In [62]:
# as_retriever() 메소드에 질문에 가장 적합한 문서를 가져올 개수(기본값은 4)를 넘겨서 벡터저장소를 벡터검색기로 만든다.
chroma_k_retirever = chroma_db.as_retriever(search_kwargs={'k': 2})

query = '리비안은 언제 사업을 시작했나요?'
# invoke() 메소드의 인수로 질문을 넘겨서 질문에 가장 적합한 문서들을 가져온다.
retirever_docs = chroma_k_retirever.invoke(query)

print(f'쿼리: {query}', end='\n\n')
print('검색 결과')
print('-' * 100)
for doc in retirever_docs:
    print(doc.page_content)
    print(doc.metadata)
    # cosine_similarity() 함수의 인수로 질문과 답변을 임베딩된 결과를 리스트 형태로 넘겨서 코사인 유사도를 계산한다.
    score = cosine_similarity(
        [embeddings_model.embed_query(query)], # 질문
        [embeddings_model.embed_query(doc.page_content)] # 답변
    )[0][0]
    print(f'코사인 유사도: {score:.4f}')
    print('-' * 100)

쿼리: 리비안은 언제 사업을 시작했나요?

검색 결과
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.6769
----------------------------------------------------------------------------------------------------
리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다.이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.5992
----------------------------------------------------------------------------------------------------


## 벡터검색기에 유사도 점수의 임계값(Threshold) 지정

`search_type` 속성은 벡터 저장소에서 문서를 `어떤 전략`으로 찾아올 것인가를 지정한다.  
`similarity`가 기본값으로 사용되며 가장 표준적인 검색 방식으로 사용자 질문과 가장 유사한 문서 조각을 유사도 점수 상위 k개를 가져온다. 엉뚱한 답을 할 수 있다.  
`similarity_score_threshold`는 설정한 임계값을 넘는 문서 조각을 유사도 점수 상위 k개를 가져온다. 임계값을 너무 높게잡으면 결과가 빈번하게 누락된다.  
`mmr`은 다양성을 고려한 검색 방식으로 질문과 유사한 문서 조각을 찾되, 이미 찾은 문서들과 내용이 너무 중복되는 문서는 제외한다.

In [63]:
chroma_threshold_retirever = chroma_db.as_retriever(
    # 설정한 임계값을 넘는 문서 조각을 유사도 점수 상위 k개를 가져온다.
    search_type='similarity_score_threshold',
    search_kwargs={
        'k': 2,
        # 코사인 유사도 점수가 0.6을 넘는 결과만 반환하도록 설정한다. 코사인 유사도는 높을수록 좋다. 낮은 유사도의 무의미한 결과는 버린다.
        # 임계값 설정을 사용하려면 search_type='similarity_score_threshold' 속성을 지정해야 한다.
        'score_threshold': 0.6
    }
)

query = '리비안은 언제 사업을 시작했나요?'
retirever_docs = chroma_threshold_retirever.invoke(query)

print(f'쿼리: {query}', end='\n\n')
print('검색 결과')
print('-' * 100)
for doc in retirever_docs:
    print(doc.page_content)
    print(doc.metadata)
    score = cosine_similarity([embeddings_model.embed_query(query)], [embeddings_model.embed_query(doc.page_content)])[0][0]
    print(f'코사인 유사도: {score:.4f}')
    print('-' * 100)

쿼리: 리비안은 언제 사업을 시작했나요?

검색 결과
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.6769
----------------------------------------------------------------------------------------------------


## 벡터검색기 MMR(Maximal Marginal Relevance) 검색

MMR은 RAG 시스템에서 검색 결과의 `관련성(Relevance)`과 `다양성(Diversity)`을 동시에 고려하여 최적의 문서 집합을 선별하는 `중복 제거` 알고리즘이다.  
단순히 쿼리와 가장 유사한 상위 k개 문서만 가져오면, 거의 비슷한 내용의 문서들이 중복으로 뽑히는 정보의 `중복성(Redundancy)` 문제가 발생합니다. MMR은 이러한 한계를 극복하기 위해 등장했다.

search_kwargs 속성에 최종적으로 사용자에게 보여줄(반환할) 문서의 개수(`k`), MMR 알고리즘을 적용하기 위해 벡터저장소에서 뽑아낼 후보군의 개수(`fetch_k`), 다양성과 관련성의 비율(`lambda_mult`)을 지정한다. 

In [64]:
chroma_mmr = chroma_db.as_retriever(
    search_type='mmr',
    search_kwargs={
        # 최종적으로 사용자에게 보여줄(반환할) 문서의 개수를 지정한다.
        'k': 3,
        # MMR 알고리즘을 적용하기 위해 벡터저장소에서 뽑아낼 후보군 개수를 지정한다.
        # 'fetch_k'에 지정한 6개 중에서 중복되지 않는 최적의 'k'에 지정한 3개를 고르게 된다. 'fetch_k' 값은 'k' 보다 크거나 같아야 한다.
        'fetch_k': 6,
        # 다양성과 관련성의 비율을 지정한다.
        # 1.0에 가까울수록 질문과의 '유사도'만 따져서 기본 검색과 비슷해지고 0.0에 가까울수록 이미 뽑힌 결과와 얼마나 다른가 즉, '다양성'을 더 중요하게 여긴다.
        'lambda_mult': 0.5 # 기본값은 0.5
    }
)

query = '리비안은 언제 사업을 시작했나요?'
retirever_docs = chroma_mmr.invoke(query)

print(f'쿼리: {query}', end='\n\n')
print('검색 결과')
print('-' * 100)
for doc in retirever_docs:
    print(doc.page_content)
    print(doc.metadata)
    score = cosine_similarity([embeddings_model.embed_query(query)], [embeddings_model.embed_query(doc.page_content)])[0][0]
    print(f'코사인 유사도: {score:.4f}')
    print('-' * 100)

쿼리: 리비안은 언제 사업을 시작했나요?

검색 결과
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.6769
----------------------------------------------------------------------------------------------------
리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다.이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.5992
----------------------------------------------------------------------------------------------------
머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다.회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다.테슬라는 2010년 6월 나스닥에 상장되었습니다.2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다.
{'source': './data\\테슬라_KR.txt'}
코사인 유사도: 0.2862
------------------

## 벡터검색기 metadata 필터링 검색

벡터검색기에서 metadata에 특정 조건(filter)을 지정해서 검색한다.

In [65]:
chroma_metadata = chroma_db.as_retriever(
    search_kwargs={
        'k': 2,
        # 'filter'에 딕셔너리 형태로 메타데이터를 검색할 조건을 지정한다. 리비안_KR.txt 문서의 데이터만 검색 대상으로 지정한다.
        'filter': {'source': './data\\리비안_KR.txt'}
    }
)

query = '리비안은 언제 사업을 시작했나요?'
retirever_docs = chroma_metadata.invoke(query)

print(f'쿼리: {query}', end='\n\n')
print('검색 결과')
print('-' * 100)
for doc in retirever_docs:
    print(doc.page_content)
    print(doc.metadata)
    score = cosine_similarity([embeddings_model.embed_query(query)], [embeddings_model.embed_query(doc.page_content)])[0][0]
    print(f'코사인 유사도: {score:.4f}')
    print('-' * 100)

쿼리: 리비안은 언제 사업을 시작했나요?

검색 결과
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.6769
----------------------------------------------------------------------------------------------------
리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다.이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.5992
----------------------------------------------------------------------------------------------------


## 벡터검색기 page_content 필터링 검색

벡터검색기에서 page_content에 특정 조건(where_document)을 지정해서 검색한다.

In [66]:
chroma_metadata = chroma_db.as_retriever(
    search_kwargs={
        'k': 2,
        # 'where_document'에 딕셔너리 형태로 page_content를 검색할 조건을 지정한다. 본문에 '리비안'이라는 단어가 포함된 데이터만 검색 대상으로 지정한다.
        # 'where_document'를 사용할 때 반드시 예약된 연산자($)를 키로 사용해야 한다.
        # $contains: 문서 내용에 특정 문자열이 포함된
        # $not_contains: 문서 내용에 특정 문자열이 포함되지 않은
        # $regex: 정규 표현식 패턴과 일치하는 경우
        # $not_regex: 정규 표현식 패턴과 일치않는 경우
        # $and / $or: 여러 조건을 결합해서 사용할 경우
        'where_document': {'$contains': '리비안'}
    }
)

query = '리비안은 언제 사업을 시작했나요?'
retirever_docs = chroma_metadata.invoke(query)

print(f'쿼리: {query}', end='\n\n')
print('검색 결과')
print('-' * 100)
for doc in retirever_docs:
    print(doc.page_content)
    print(doc.metadata)
    score = cosine_similarity([embeddings_model.embed_query(query)], [embeddings_model.embed_query(doc.page_content)])[0][0]
    print(f'코사인 유사도: {score:.4f}')
    print('-' * 100)

쿼리: 리비안은 언제 사업을 시작했나요?

검색 결과
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.6769
----------------------------------------------------------------------------------------------------
리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다.이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.5992
----------------------------------------------------------------------------------------------------


# 키워드 기반 검색(Keyword Search) - BM25Retriever

사용자가 입력한 단어나 구문을 문서 내에서 직접 찾는 전통적인 검색 방법이다.

BM25는 키워드 검색을 효과적을 만드는 알고리즘 중 하나로 TF-IDF(Term Frequency - Inverse Document Frequency)의 한계를 보완한 랭킹 함수이다.  
용어 빈도(TF)가 증가함에 따라 점수 증가율이 감소하며 이는 특정 단어가 과도하게 반복되는 경우 영향을 제한하고 문서 길이 정규화를 통해 긴 문서에서 용어가 더 자주 나타날 가능성을 고려하여 조정한다.

장점  
&nbsp;&nbsp;&nbsp;▶ 단순하면서도 효과적인 랭킹 시스템이다.  
&nbsp;&nbsp;&nbsp;▶ 계산 비용이 상대적으로 낮다.  
&nbsp;&nbsp;&nbsp;▶ 특정 키워드나 구문 검색에 효과적이다.  
단점  
&nbsp;&nbsp;&nbsp;▶ 의미적 유사성을 전혀 고려하지 않는다.  
&nbsp;&nbsp;&nbsp;▶ 동의어 관련어를 자동으로 처리하지 못한다.  
&nbsp;&nbsp;&nbsp;▶ 문맥을 전혀 이해하지 못한다.

키워드 기반의 전통적이고 강력한 검색 알고리즘은 BM25를 사용하기 위해 BM25Retriever를 import 한다.

In [67]:
from langchain_community.retrievers import BM25Retriever

In [68]:
chroma_db.get().keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

벡터저장소에서 BM25Retriever로 검색할 데이터를 준비한다.

In [69]:
# 벡터저장소에서 원본 데이터 추출
documents = chroma_db.get()['documents'] # 벡터저장소에 저장된 본문을 리스트 형태로 추출한다.
metadatas = chroma_db.get()['metadatas'] # 벡터저장소에 저장된 부가 정보를 리스트 형태로 추출한다.
# print(documents, metadatas)

# 추출한 원본 데이터로 Document 객체 생성
# 리스트 컴프리헨션을 사용하여, 단순 텍스트였던 데이터들을 LangChain이 인식할 수 있는 Document 객체들이 저장된 리스트를 만든다.
docs = [Document(page_content=document, metadata=metadata) for document, metadata in zip(documents, metadatas)]
# print(docs)
print(len(docs))

6


준비된 데이터로 BM25Retriever 검색기를 생성한다.

In [70]:
# from_documents() 메소드로 준비된 데이터와 검색할 문서 개수를 넘겨 키워드 검색용 색인(index)을 만든다.
bm25_retriever = BM25Retriever.from_documents(docs, k=4)
bm25_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001A960FAEE10>)

In [71]:
query = '리비안은 언제 사업을 시작했나요?'
retirever_docs = bm25_retriever.invoke(query)

print(f'쿼리: {query}', end='\n\n')
print('검색 결과')
print('-' * 100)
for doc in retirever_docs:
    print(doc.page_content)
    print(doc.metadata)
    score = cosine_similarity([embeddings_model.embed_query(query)], [embeddings_model.embed_query(doc.page_content)])[0][0]
    print(f'코사인 유사도: {score:.4f}')
    print('-' * 100)

쿼리: 리비안은 언제 사업을 시작했나요?

검색 결과
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.6769
----------------------------------------------------------------------------------------------------
리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다.2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.
{'source': './data\\리비안_KR.txt'}
코사인 유사도: 0.5080
----------------------------------------------------------------------------------------------------
2012년부터 2023년 3분기까지 테슬라의 전 세계 누적 판매량은 4,962,975대를 초과했습니다.SMT Packaging에 따르면, 2023년 테슬라의 판매량은 전 세계 전기차 시장의 약 12.9%를 차지했습니다.
{'source': './data\\테슬라_KR.txt'}
코사인 유사도: 0.1633
-----------------------------------------------

BM25 알고리즘이 내부적으로 각 문서에 점수를 매기는 과정을 알아본다.

In [72]:
# 질문을 토큰화(단어 분리) 한다.
query = '리비안은 언제 사업을 시작했나요?'
tokenized_query = query.split()
print(tokenized_query)

# 문서별 점수를 계산한다.
# BM25 검색기 내부의 단어 빈도 등의 통계 모델(vectorizer)에 접근해서 get_scores()의 인수로 질문을 분리한 단어들을 바탕으로 각 문서의 연관성 점수를 계산한다.
doc_scores = bm25_retriever.vectorizer.get_scores(tokenized_query)
print(doc_scores)

# 점수가 높은 순으로 정렬한다.
# sorted() 함수의 key 속성으로 정렬할 데이터가 리스트나 튜플로 구성된 형태일 경우 리스트나 튜플의 특정 요소를 기준으로 정렬하라고 지정할 수 있다.
doc_scores_sorted = sorted(enumerate(doc_scores), reverse=True, key=lambda x: x[1])
doc_scores_sorted

['리비안은', '언제', '사업을', '시작했나요?']
[0.80625604 0.         0.54069396 0.         0.         0.        ]


[(0, 0.806256044777243),
 (2, 0.5406939647467565),
 (1, 0.0),
 (3, 0.0),
 (4, 0.0),
 (5, 0.0)]

In [73]:
# query = '리비안이 설립된 연도는?'
query = '리비안이 첫 양산 차량을 생산한 년도는?'
retirever_docs = bm25_retriever.invoke(query)

print(f'쿼리: {query}', end='\n\n')
print('검색 결과')
print('-' * 100)
for doc in retirever_docs:
    print(doc.page_content)
    print('-' * 100)

쿼리: 리비안이 첫 양산 차량을 생산한 년도는?

검색 결과
----------------------------------------------------------------------------------------------------
리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다.2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.
----------------------------------------------------------------------------------------------------
머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다.회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다.테슬라는 2010년 6월 나스닥에 상장되었습니다.2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다.
----------------------------------------------------------------------------------------------------
2012년부터 2023년 3분기까지 테슬라의 전 세계 누적 판매량은 4,962,975대를 초과했습니다.SMT Packaging에 따르면, 2023년 테슬라의 판매량은 전 세계 전기차 시장의 약 12.9%를 차지했습니다.
----------------------------------------------------------------------------------------------------
테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다.2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설

In [74]:
# query = '리비안이 설립된 연도는?'
query = '리비안이 첫 양산 차량을 생산한 년도는?'
tokenized_query = query.split()
# print(tokenized_query)
doc_scores = bm25_retriever.vectorizer.get_scores(tokenized_query)
# print(doc_scores)
doc_scores_sorted = sorted(enumerate(doc_scores), reverse=True, key=lambda x: x[1])
doc_scores_sorted

[(2, 2.390372255669916),
 (4, 1.3381319770112319),
 (0, 0.0),
 (1, 0.0),
 (3, 0.0),
 (5, 0.0)]

In [75]:
# documents

# 키워드 기반 검색(Keyword Search) - BM25Retriever + Kiwi 한국어 분석기

`Kiwi 한국어 분석기`를 사용하려면 `pip install kiwipiepy`를 실행해서 kiwipiepy 라이브러리를 설치하고 Kiwi를 import 한다.

Kiwi 한국어 분석기는 `numpy 1.x 버전`을 사용해서 작성된 라이브러리로 아직 `numpy 2.x 버전을 지원하지 않는다.`  
2026-09-11 현재 numpy 설치하면 2.4.6 버전이 설치된다. numpy가 설치된 상태에서 kiwipiepy를 설치하면 자동으로 numpy가 1.26.4 버전으로 다운그레이드되서 설치되지만 kiwipiepy를 설치한 후 numpy 다시 설치하면 2.4.6 버전이 설치되서 에러가 발생한다.  
kiwipiepy를 다시 설치하지 않고 numpy만 다운그레이드 하려면 `pip install numpy==1.26.4` 명령을 실행해서 다운그레이드 한다.

In [76]:
from kiwipiepy import Kiwi

Kiwi 형태소 분석기를 사용해 한국어 문장을 단어 단위로 나누고 토큰화하는 전처리 함수를 정의한다.

In [77]:
def bm25_preprocess_func(text):
    # Kiwi 분석기 객체를 생성한다.
    kiwi_model = Kiwi()
    # add_user_word() 메소드로 단어와 품사를 넘겨서 단어 사전에 사용자 단어를 추가할 수 있다.
    # '리비안'을 고유 명사로 사전에 등록한다. 이렇게 하면 '리비아', 'ㄴ'처럼 쪼개지지 않고 하나의 단어로 정확하게 인식된다.
    # '리비안'이라는 단어가 고유명사(NNP)로 단어 사전에 추가된다.
    kiwi_model.add_user_word('리비안', 'NNP')
    # '테슬라'라는 단어가 고유명사(NNP)로 단어 사전에 추가된다.
    kiwi_model.add_user_word('테슬라', 'NNP')
    # 입력받은 문장(text)을 토큰화 한 뒤, 형태소의 글자 모양만 리스트 형태로 반환한다. '리비안이' => ['리비안', '이']
    # tokenize() 메소드의 인수로 문장을 넘겨셔서 형태소 분석을 실행한다.
    # print(kiwi_model.tokenize(text))
    return [token.form for token in kiwi_model.tokenize(text)]

In [78]:
bm25_preprocess_func('리비안이 설립된 연도는')

['리비안', '이', '설립', '되', 'ᆫ', '연도', '는']

준비된 데이터로 한글 형태소 분석을 하는 bm25_preprocess_func() 함수를 사용해서 BM25Retriever 검색기를 생성한다.

In [79]:
bm25_retriever = BM25Retriever.from_documents(
    documents=docs,
    k=2,
    # 위에서 만든 토큰화 전처리를 bm25_preprocess_func 함수를 호출해서 실행한다.
    preprocess_func=bm25_preprocess_func
)

In [80]:
# query = '리비안이 설립된 연도는?'
query = '리비안이 첫 양산 차량을 생산한 년도는?'
retirever_docs = bm25_retriever.invoke(query)

print(f'쿼리: {query}', end='\n\n')
print('검색 결과')
print('-' * 100)
for doc in retirever_docs:
    print(doc.page_content)
    print('-' * 100)

쿼리: 리비안이 첫 양산 차량을 생산한 년도는?

검색 결과
----------------------------------------------------------------------------------------------------
리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다.2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.
----------------------------------------------------------------------------------------------------
리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다.이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다.
----------------------------------------------------------------------------------------------------


In [81]:
# query = '리비안이 설립된 연도는?'
query = '리비안이 첫 양산 차량을 생산한 년도는?'

tokenized_query = bm25_preprocess_func(query)
print(tokenized_query)

doc_scores = bm25_retriever.vectorizer.get_scores(tokenized_query)
doc_scores_sorted = sorted(enumerate(doc_scores), reverse=True, key=lambda x: x[1])
doc_scores_sorted

['리비안', '이', '첫', '양산', '차량', '을', '생산', '하', 'ᆫ', '년', '도', '는', '?']


[(2, 6.158843269590649),
 (1, 3.1531527739571272),
 (4, 2.7284324017922756),
 (3, 1.7494853868469742),
 (0, 1.7226429184029857),
 (5, 0.7946064975009759)]

In [82]:
# documents

# 혼합 검색(Hybrid Search) - EnsembleRetriever

의미 기반 검색과 키워드 기반 검색을 결합한 방식으로 의미 기반 검색과 키워드 기반 검색을 결합하여 보다 정확하고 관련성 높은 결과를 제공한다.  
키워드 매칭과 의미적 유사성을 동시에 고려해서 다양한 관점의 검색 결과를 제공할 수 있다.

<img src="./hybridSearch.png" width="800" align="left" />

여러 검색기를 조합해 결과를 하나로 합쳐주는 혼합 검색을 하기 위해서 EnsembleRetriever를 import 한다.

In [83]:
from langchain.retrievers import EnsembleRetriever

서로 다른 검색기인 의미 기반 검색(Chroma)과 키워드 기반 검색(BM25)를 하나로 합쳐서 더 정확한 결과를 내는 앙상블 검색기를 만든다.

In [84]:
# 사용할 검색기들을 리스트로 묶어준다.
ensemble_retirevers = [chroma_threshold_retirever, bm25_retriever]
# EnsembleRetriever 클래스의 생성자로 사용할 검색기들을 묶어놓은 리스트를 넘기고 각 검색기의 결과에 줄 가중치를 지정해서 앙상블 검색기 객체를 만든다.
ensemble_retirever = EnsembleRetriever(
    # 위에서 묶은 두 검색기를 검색기로 설정한다.
    retrievers=ensemble_retirevers,
    # 위에서 묶은 두 검색기의 검색 결과에 둘 가중치를 지정한다.
    weights=[0.5, 0.5]
)

In [85]:
# query = '리비안이 설립된 연도는?'
query = '리비안이 첫 양산 차량을 생산한 년도는?'
retirever_docs = ensemble_retirever.invoke(query)

print(f'쿼리: {query}', end='\n\n')
print('검색 결과')
print('-' * 100)
for doc in retirever_docs:
    print(doc.page_content)
    print('-' * 100)

쿼리: 리비안이 첫 양산 차량을 생산한 년도는?

검색 결과
----------------------------------------------------------------------------------------------------
리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다.이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다.
----------------------------------------------------------------------------------------------------
리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다.2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.
----------------------------------------------------------------------------------------------------


# 검색 성능 평가

# 테스트셋 합성

## QA(Question-Answer) 데이터셋 합성

문서 준비  
&nbsp;&nbsp;&nbsp;▶ 관련 도메인의 문서들을 수집한다.  
&nbsp;&nbsp;&nbsp;▶ 이 문서들은 RAG 시스템의 지식 베이스 역할을 한다.  
질문 생성  
&nbsp;&nbsp;&nbsp;▶ LLM을 사용하여 각 문서에 대한 다양한 질문을 생성한다.  
&nbsp;&nbsp;&nbsp;▶ 다양한 난이도와 유형의 질문을 포함한다.(사실 기반, 추론 기반, 요약 등)  
답변 생성  
&nbsp;&nbsp;&nbsp;▶ 생성된 질문들에 대한 답변을 LLM을 사용해서 생성  
&nbsp;&nbsp;&nbsp;▶ 이 답변들을 '정답'으로 간주한다.  
메타데이터 추가  
&nbsp;&nbsp;&nbsp;▶ 각 QA 쌍에 대해 출처, 유형, 난이도 등의 메타데이터를 추가한다.

## 합성 데이터에 대한 검증 및 수정

자동화된 검증  
&nbsp;&nbsp;&nbsp;▶ 질문과 답변의 길이, 형식 등을 확인한다.  
&nbsp;&nbsp;&nbsp;▶ 답변이 질문과 관련이 있는지 간단한 관련성 검사를 한다.  
사람이 검토  
&nbsp;&nbsp;&nbsp;▶ 샘플링된 QA 쌍을 인간 전문가가 검토한다.  
&nbsp;&nbsp;&nbsp;▶ 질문의 품질, 답변의 정확성, 난이도 등을 평가한다.  
반복적 개선  
&nbsp;&nbsp;&nbsp;▶ 검토 결과를 바탕으로 생성 프롬프트를 개선한다.  
&nbsp;&nbsp;&nbsp;▶ 필요한 경우 특정 QA 쌍을 수동으로 수정한다.
다양성 확보  
&nbsp;&nbsp;&nbsp;▶ 질문 유형, 난이도, 주제 등이 균형있게 분포되어 있는지 확인한다.  
편향성 검사  
&nbsp;&nbsp;&nbsp;▶ 생성된 데이터셋이 편향이 없는지 검토한다.

<img src="./dataset.png" width="600" align="left" />

## 데이터 준비

In [86]:
docs[0].metadata['source']

'./data\\리비안_KR.txt'

os.path.join('경로명', '파일명')는 경로명과 파일명을 합쳐서 전체 경로를 만들지만 os.path.split('전체 경로')는 전체 경로를 구성하는 '경로명'과 '파일명'을 분리해서 튜플로 리턴한다.

In [87]:
os.path.split('./data\\리비안_KR.txt')[1].split('_')[0]

'리비안'

메타데이터에 고유 ID를 부여하고, 본문 내용을 가공하여 새로운 문서 리스트를 만든다.

In [88]:
docs

[Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다.2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다.주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.'),
 Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다.이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다.'),
 Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다.2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.'),
 Document(metadata={'source': './data\\테슬라_KR.txt'}, page_content='테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다.2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다.'),
 Document(metadata={'source': './data\\테슬라_KR.txt'}, page_cont

In [89]:
final_docs = []

for index, doc in enumerate(docs):
    # print(index, doc)
    # 원본 데이터를 유지하기 위해 원본 데이터를 복사해서 사본을 만든다. 깊은 복사
    # model_copy() 메소드는 RAG 파이프라인이나 문서 처리 과정에서 원본 Document 객체를 훼손하지 않고 일부 속성을 수정하여 새로운 Document 객체를 생성한다. 
    new_doc = doc.model_copy()
    # 복제된 문서 메타데이터에 'doc_id'라는 항목을 만들고 index를 저장한다.
    new_doc.metadata['doc_id'] = index
    # 복제된 문서 본문에 구두점(.!?)을 '\n'으로 바꾼다.
    # re 라이브러리의 sub() 메소드의 인수로 정규식 패턴, 바꿀 문자열, 정규식 패턴을 검색할 문자열을 넘겨서 정규식 패턴을 찾아서 바꿀 문자열로 치환한다.
    new_doc.page_content = re.sub(r'[.!?]\s*', '\n', new_doc.page_content)
    # 파일 경로(source)에서 회사명만 가져온다.
    corp_name = os.path.split('./data\\리비안_KR.txt')[1].split('_')[0]
    # LLM이 답변할 때 출처를 더 잘 인식하도록 돕기 위해서 문서 본문 아래에 회사명을 포함한 안내 문구를 추가한다.
    new_doc.page_content = f'{new_doc.page_content}\n(참고: 이 문서는 {corp_name}에 대한 정보를 담고 있습니다.)'
    # print(new_doc)
    final_docs.append(new_doc)
    
for doc in final_docs[:2]:
    print(doc.page_content)
    print('-' * 100)
    print(doc.metadata)
    print('=' * 100 + '\n')

리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다
2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다
주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
----------------------------------------------------------------------------------------------------
{'source': './data\\리비안_KR.txt', 'doc_id': 0}

리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다
이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
----------------------------------------------------------------------------------------------------
{'source': './data\\리비안_KR.txt', 'doc_id': 1}



위에서 가공된 문서 리스트(final_docs)를 JSONL(JSON Line) 파일로 저장한다.  
JSONL은 한 줄에 하나의 JSON 객체를 기록하는 방식으로, 대용량 데이터를 처리하거나 LLM 학습용 데이터를 저장할 때 표준처럼 사용된다.

In [90]:
final_docs[0]

Document(metadata={'source': './data\\리비안_KR.txt', 'doc_id': 0}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다\n2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다\n주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)')

In [91]:
# './data' 폴더의 'final_docs.jsonl'을 바이너리 형태의 출력용으로 open 한다.
# 'wb'는 Write Binary로 파일을 '쓰기 전용' 및 '바이너리 모드'로 열어서 텍스트가 아닌 바이트 단위로 저장하겠다는 의미이다.
# with 구문을 사용했으므로 with 블록의 모든 작업이 완료되면 파일이 자동으로 안전하게 닫힌다.
with open('./data/final_docs.jsonl', 'wb') as file:
    # 앞서 만든 문서 리스트에 저장된 문서 객체들을 하나씩 꺼내서 파일로 출력한다.
    for doc in final_docs:
        # dict(doc)는 doc에 저장된 Document 객체를 파이썬의 딕셔너리로 변환한다.
        # dumps() 메소드는 인수로 지정된 데이터(문자열, 리스트, 딕셔너리 등)를 JSON 형태로 변환한다.
        # 한글이 유니코드로 표현되지 않고 한글을 그대로 유지하기 위해서 ensure_ascii=False 속성을 지정한다.
        # encode('utf-8')는 생성되는 JSON 문자열을 UTF-8 형식의 바이트 데이터로 변환한다. 파일 열기 모드가 'wb'이므로 이 과정이 반드시 필요하다.
        file.write(json.dumps(dict(doc), ensure_ascii=False).encode('utf-8'))
        # 한 데이터 작성이 끝날 때마다 줄바꿈 기호를 추가한다. 앞에 'b'를 붙여서 바이트 형태로 저장한다.
        file.write(b'\n')

JSONL 파일을 읽어서 Document 객체로 불러온다.

JSONL 파일의 각 줄에서 어떤 데이터를 메타데이터로 사용할지 정의하는 함수를 만든다.

In [92]:
def metadata_func(record, metadata):
    # print(metadata)
    # JSONLoader가 JSON 또는 JSONL 파일을 읽을때 자동으로 생성되는 메타데이터를 실제 JSON 또는 JSONL 파일에서 읽어들인 메타데이터로 
    metadata = record['metadata']
    return metadata

jq_schema 속성은 JSON 데이터 내에서 특정 필드나 구조를 선택해서 Document 객체로 변환하기 위해 사용하는 필터링 규칙이다.

`JSON 데이터 예시              |  jq_schema 설정  |  설명                                                                                     `  
`{"text": "안녕하세요"}        |  '.text'         |  text 키의 값만 추출한다.                                                                 `  
`[{"msg": "A"}, {"msg": "A"}]  |  '.[]'           |  배열 내부의 각 객체를 별도의 Document로 생성한다.                                        `  
`[{"msg": "A"}, {"msg": "A"}]  |  '.[].msg        |  배열 내부의 각 객체의 msg 속성값만 추출해서 Document로 생성한다.                         `  
`{"data": {"content": ...}}    |  '.data.content  |  중첩된 경로(data 키에 할당된 값 중에서 content 키에 접근한다)를 따라서 데이터를 추출한다.`

In [93]:
json_loader = JSONLoader(
    file_path='./data/final_docs.jsonl',
    jq_schema='.', # '.'는 JSONL 데이터의 한 줄을 의미한다.
    content_key='page_content', # JSONL 데이터에서 중에서 읽어들일 키를 지정한다.
    json_lines=True, # 읽어들이는 파일이 JSON 파일이 아니라 JSONL 파일 형식임을 알려준다.
    # 메타데이터를 읽어들이는 콜백 함수를 지정한다.
    # 콜백 함수의 첫 번째 인수로 읽어들이는 JSONL 파일의 데이터 전체가 넘어가고 두 번째 인수로 JSONLoader로 읽어들일 때 기본 메타데이터가 넘어간다.
    metadata_func=metadata_func
)

json_docs = json_loader.load()

print(f'읽어들인 문서의 수: {len(json_docs)}')
print('-' * 100)
for doc in json_docs[:2]:
    print(doc.page_content)
    print(doc.metadata)
    print('-' * 100)

읽어들인 문서의 수: 6
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다
2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다
주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
{'source': './data\\리비안_KR.txt', 'doc_id': 0}
----------------------------------------------------------------------------------------------------
리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다
이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
{'source': './data\\리비안_KR.txt', 'doc_id': 1}
----------------------------------------------------------------------------------------------------


JSONL 파일에서 읽어들인 데이터를 판다스 데이터프레임으로 변환한다.

In [94]:
# Document 객체에서 필요한 정보만 뽑아서 딕셔너리로 만든 다음에 저장할 빈 리스트를 선언한다.
test_data = []

for doc in json_docs:
    # 각 문서에서 원하는 정보를 꺼내서 딕셔너리 형태로 만들어서 test_data에 추가한다.
    test_data.append({
        'context': doc.page_content,
        # 딕셔너리에서 특정 키의 데이터를 얻어올 때 ['key'] 형태로 얻어오면 딕셔너리에 'key'가 존재하지 않으면 에러가 발생된다.
        # 딕셔너리에 'key'가 존재하지 않아도 에러가 발생되지 않게 하려면 get('key') 형태를 사용하면 된다.
        # get() 메소드의 두 번째 인수로 'key'가 존재하지 않을 때 'None'대신 채울 기본값을 지정할 수 있다.
        'source': doc.metadata.get('source', ''),
        'doc_id': doc.metadata['doc_id']
    })

df_test = pd.DataFrame(test_data)
print(df_test.shape)
df_test

(6, 3)


,context,source,doc_id
0,리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차...,./data\리비안_KR.txt,0
1,"리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드...",./data\리비안_KR.txt,1
2,"리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버...",./data\리비안_KR.txt,2
3,"테슬라(Tesla, Inc\n)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기...",./data\테슬라_KR.txt,3
4,머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다\n회사 이름은...,./data\테슬라_KR.txt,4
5,"2012년부터 2023년 3분기까지 테슬라의 전 세계 누적 판매량은 4,962,97...",./data\테슬라_KR.txt,5


pydantic을 사용해서 문서로 부터 구조화된 질의응답(QA) 쌍을 생성한다.

AI 모델의 답변(LLM이 생성한 자연어 텍스트)을 Pydantic에서 정의한 구조(클래스로 만든 객체)로 변환하는 파서를 사용하기 위해 PydanticOutputParser를 import 한다.  
데이터의 형태(스키마)를 정의하고 validation을 수행하기 위해 BaseModel, Field를 import 한다.  
파이썬 표준 라이브러리의 모듈로, 데이터의 타입에 대한 힌트를 주기 위해 import 한다.  

In [95]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

AI 모델의 자유로운 답변을 우리가 원하는 '규격화된 데이터' 형태로 강제하기 위한 스키마(데이터 구조 => 클래스(설계도))를 정의한다.

질문과 답변 한 쌍의 구조, 이 클래스는 '질문 하나와 그에 대한 답변 하나'가 어떻게 생성되어야 하는지 정의한다.  

In [96]:
# PydanticOutputParser에서 사용되는 클래스는 반드시 pydantic의 BaseModel 클래스를 상속받아 만든다.
class QAPair(BaseModel):
    # 독스트링(docstring)으로 클래스의 설명이다.
    # AI 프레임워크와 연결될 때 AI 모델에게 이 클래스가 전체적으로 무엇을 의미하는지 안내하는 힌트(prompt)역할을 한다.
    '''질문과 답변을 한 쌍으로 저장한다.'''
    # 'question: str'는 question 변수를 정의하며, 저장되는 데이터는 반드시 문자열(str)이어야 함을 명시한다.
    # '...'은 이 값이 필수 입력 항목임을 뜻한다. 데이터가 입력될 때 question 값이 누락되면 에러가 발생된다.
    # description 속성은 필드에 대한 설명 메타 데이터로 AI가 데이터를 분석해서 JSON 데이터로 변환할 때, 어떤 정보를 할당할지 판단하는 가이드라인으로 활용된다.
    question: str = Field(..., description='''생성된 질문(질문을 한국어로 작성해 주세요.)''')
    # 'answer: str'는 answer 변수를 정의하며, 저장되는 데이터는 반드시 문자열(str)이어야 함을 명시한다.
    answer: str = Field(..., description='''질문에 대한 답변(질문의 핵심 내용을 반영해서 사실 기반 질문에 대한 답변을 한국어로 작성해 주세요.)''')

질문과 답변 여러 개의 쌍을 묶는 리스트 구조, 이 클래스는 질문-답변 쌍들의 집합을 정의한다.

In [97]:
class QASet(BaseModel):
    '''질문과 답변을 한 쌍으로 묶은 여러 질문과 답변을 저장한다.'''
    # 'qa_pairs: List'는 qa_pairs 변수를 정의하며, 저장되는 데이터는 반드시 QAPair 클래스 객체가 저장된 List이어야 함을 명시한다.
    qa_pairs: List[QAPair] = Field(..., description='''질문-답변이 한 쌍으로 정의된 QAPair 클래스 객체가 저장된 리스트''')

AI가 수행해야 할 업무의 '가이드 라인'을 정의하는 프롬프트 템플릿을 정의한다.

① 당신은 수행할 과제는 ...: AI에게 `역할을 부여`한다.  
② 각 사실 기반 질문은 ...: 상상해서 답변하지 말고, 반드시 주어진 본문 속의 객관적인 정보만 사용하라는 `제약 조건`이다.  
③ 질문은 사용자들이 검색 엔진 ...: 질문을 'OO의 특징은?' 처럼 핵심 키워드 중심으로 `짧고 명확`하게 만들라는 `지시 사항`이다.  
④ 질문에 '문맥에 근거하여'...: AI가 흔히 사용하는 상투적인 문구를 제거하여, `자연스러운 질문을 유도`하는 `지시 사항`이다.  
⑤ 명확하고 완전한 정보를 ...: 단순히 '네', '아니오'가 아니라, 답변만 읽어도 무슨 내용인지 알 수 있도록 `완성된 문장`을 만들라는 `지시 사항`이다.  

In [98]:
QA_generation_template = '''
당신은 수행할 과제는 제공된 문맥(context)을 바탕으로 {num_questions}개의 사실 기반 질문-답변 쌍을 만드는 것입니다.
각 사실 기반 질문은 문맥(context)에 나오는 구체적이고 간결한 사실 정보로 답변할 수 있어야 합니다.
질문은 사용자들이 검색 엔진을 사용할 때 쓰는 스타일로 구성하세요.
질문에 '문맥에 근거하여' 또는 '지문에 다르면'와 같은 표현은 포함하지 마세요.
명확하고 완전한 정보를 제공할 수 있도록 답변에 질문의 핵심 내용을 포함하세요.

문맥(context)을 제공합니다.
문맥(context): {context}

출력은 다음 형식으로 제공해 주세요.
{format_instructions}
'''

프롬프트의 변수({num_questions}, {context}, {format_instructions})를 채워서 프롬프트를 완성하고 완성된 프롬프트로 던진 질문에 대해서 응답하는 AI 모델을 정의하고 AI 모델이 응답한 결과를 정해진 데이터 형식으로 만들어주는 PydanticOutputParser를 정의한 다음 체인으로 연결한다.

In [99]:
# AI의 응답을 QASet 클래스에서 정의한 구조로 바꾸는 PydanticOutputParser를 정의한다.
# PydanticOutputParser 클래스의 생성자로 QASet 클래스를 넘겨서 AI의 답변이 JSON 형식인지, 필요한 필드(question, answer)가 다 있는지 검사하고 객체를 만든다.
pydantic_parser = PydanticOutputParser(pydantic_object=QASet)

In [100]:
# 프롬프트 템플릿을 생성한다. AI에게 줄 최종 지시서 양식을 완성한다.
QA_generation_prompt = ChatPromptTemplate.from_template(
    template=QA_generation_template,
    # template의 {format_instructions} 변수에 '반드시 JSON 형식으로 대답하고, 이러저러한 규칙을 지켜라'라는 PydanticOutputParser의 지시사항을 채워넣는다.
    # 이렇게 해주면 나중에 문맥({context})만 넣어주면 된다.
    partial_variables={'format_instructions': pydantic_parser.get_format_instructions()}
)

In [101]:
# AI 모델을 정의한다.
QA_generator = ChatOpenAI(
    model='gpt-4o-mini', # OpenAI 모델 중 가성비가 좋고 속도가 빠른 모델을 사용한다.
    temperature=0.3, # 창의성을 조절한다. 0.3 정도면 낮은 수치로, AI가 마음대소 상상하지 않고 제시된 사실에 기반해서 일관성 있는 답변을 하도록 유도한다.
    max_completion_tokens=500
)

In [102]:
# |(파이프 연산자)를 사용해서 프롬프트, AI 모델, PydanticOutputParser를 체인으로 연결한다.
# 입력받은 데이터를 바탕으로 프롬프트(QA_generation_prompt)를 만들고 완성된 프롬프트를 AI 모델(QA_generator)에게 전달해서 답변을 받고 AI의 답변을 
# PydanticOutputParser(pydantic_parser)에 넘겨서 우리가 원하는 형태(QASet 클래스)로 변환한다.
QA_generator_chain = QA_generation_prompt | QA_generator | pydantic_parser

위에서 정의한 RAG chain(QA_generator_chain)을 실제로 가동하여, 특정 본문의 질의의답 쌍을 뽑아낸다.

In [103]:
# 컨텍스트로 사용할 데이터
test_context = df_test.context[0]
print('컨텍스트:\n', test_context, sep='')

컨텍스트:
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다
2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다
주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)


In [104]:
# QA 테스트 데이터셋 생성 함수를 정의한다.
# 이 함수는 컨텍스트(본문)과 생성할 질문의 개수를 받아서 RAG chain을 실행해서 AI에게 던져주고 결과를 받는다.
# '-> QASet'의 의미는 이 함수를 실행하면 최종적으로 QASet 클래스에서 정의한 형태의 데이터가 리턴된다는 의미이다.
def generate_qa_dataset(context: str, num_questions: int) -> QASet:
    # RAG chain을 실행할 때 프롬프트 템플릿의 비어있던 변수({context}, {num_questions})를 채워서 실행한다.
    response = QA_generator_chain.invoke({
        'context': context,
        'num_questions': num_questions
    })
    return response

In [105]:
QA_set = generate_qa_dataset(test_context, 2)
print('생성된 QA 쌍')
for qa_pair in QA_set.qa_pairs:
    print(f'질문: {qa_pair.question}')
    print(f'답변: {qa_pair.answer}')
    print('-' * 100)

생성된 QA 쌍
질문: 리비안은 언제 설립되었나요?
답변: 리비안은 2009년에 설립되었습니다.
----------------------------------------------------------------------------------------------------
질문: 리비안의 본사는 어디에 위치하고 있나요?
답변: 리비안의 본사는 미시간주 리보니아에 위치하고 있습니다.
----------------------------------------------------------------------------------------------------


In [146]:
NUM_QUESTIONS = 3
output = []

# iterrows() 메소드로 데이터프레임의 데이터를 한 줄씩 꺼내서 반복시킨다.
for row in df_test.iterrows():
    # row[0]은 인덱스이고 row[1]은 해당 행의 실제 데이터 이다.
    # print(row[1])
    qa_set = generate_qa_dataset(row[1].context, NUM_QUESTIONS)
    # print(qa_set)
    
    for qa_pair in qa_set.qa_pairs:
        output.append({
            'context': [row[1].context],
            'source': [row[1].source],
            'doc_id': [row[1].doc_id],
            'question': qa_pair.question,
            'answer': qa_pair.answer,
        })
print(output)

[{'context': ['리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다\n2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다\n주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)'], 'source': ['./data\\리비안_KR.txt'], 'doc_id': [0], 'question': '리비안은 언제 설립되었나요?', 'answer': '리비안은 2009년에 설립되었습니다.'}, {'context': ['리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다\n2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다\n주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)'], 'source': ['./data\\리비안_KR.txt'], 'doc_id': [0], 'question': '리비안의 본사는 어디에 있나요?', 'answer': '리비안의 본사는 미시간주 리보니아에 있습니다.'}, {'context': ['리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다\n2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다\n주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)'], 'source': ['./data\\리비안_KR.txt'], 'doc_id': [0], 'question': '리비안은 어떤 종류의 차량에 

In [147]:
df_test_qa = pd.DataFrame(output)
print(df_test_qa.shape)
df_test_qa

(18, 5)


,context,source,doc_id,question,answer
0,[리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기...,[./data\리비안_KR.txt],[0],리비안은 언제 설립되었나요?,리비안은 2009년에 설립되었습니다.
1,[리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기...,[./data\리비안_KR.txt],[0],리비안의 본사는 어디에 있나요?,리비안의 본사는 미시간주 리보니아에 있습니다.
2,[리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기...,[./data\리비안_KR.txt],[0],리비안은 어떤 종류의 차량에 집중하고 있나요?,리비안은 자율 전기차에 집중하고 있습니다.
3,"[리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미...",[./data\리비안_KR.txt],[1],리비안의 초기 모델 R1의 원래 이름은 무엇인가요?,리비안의 초기 모델 R1의 원래 이름은 Avera입니다.
4,"[리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미...",[./data\리비안_KR.txt],[1],리비안 R1의 디자인을 맡은 디자이너는 누구인가요?,리비안 R1의 디자인은 피터 스티븐스가 맡았습니다.
5,"[리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미...",[./data\리비안_KR.txt],[1],리비안 R1은 어떤 구조적 특징을 가지고 있나요?,리비안 R1은 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 합니다.
6,"[리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 ...",[./data\리비안_KR.txt],[2],리비안은 어떤 버전을 고려했나요?,"리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버..."
7,"[리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 ...",[./data\리비안_KR.txt],[2],리비안의 첫 번째 양산 차량은 무엇인가요?,리비안의 첫 번째 양산 차량은 R1T 트럭입니다.
8,"[리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 ...",[./data\리비안_KR.txt],[2],리비안은 언제 첫 번째 양산 차량을 고객에게 인도하기 시작했나요?,리비안은 2021년 10월에 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 ...
9,"[테슬라(Tesla, Inc\n)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전...",[./data\테슬라_KR.txt],[3],테슬라는 어디에 본사를 두고 있나요?,테슬라는 텍사스주 오스틴에 본사를 두고 있습니다.


## 데이터프레임을 엑셀 파일로 저장하기

위의 결과인 질문-답변 데이터프레임을 엑셀 파일로 저장한다.

In [148]:
# to_excel() 메소드의 인수로 저장할 엑셀 파일의 경로와 이름을 념겨서 엑셀 파일로 저장할 수 있다.
# index=False 속성을 지정하면 데이터프레임의 인댁스는 엑셀 파일에 저장하지 않는다.
df_test_qa.to_excel('./data/df_test_qa.xlsx', index=False)

## 엑셀 파일을 데이터프레임으로 가져오기

In [149]:
# read_excel() 메소드의 인수로 읽어올 엑셀 파일의 경로와 이름을 념겨서 데이터프레임으로 읽어올 수 있다.
df_test_qa = pd.read_excel('./data/df_test_qa.xlsx')
df_test_qa

,context,source,doc_id,question,answer
0,['리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전...,['./data\\리비안_KR.txt'],[0],리비안은 언제 설립되었나요?,리비안은 2009년에 설립되었습니다.
1,['리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전...,['./data\\리비안_KR.txt'],[0],리비안의 본사는 어디에 있나요?,리비안의 본사는 미시간주 리보니아에 있습니다.
2,['리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전...,['./data\\리비안_KR.txt'],[0],리비안은 어떤 종류의 차량에 집중하고 있나요?,리비안은 자율 전기차에 집중하고 있습니다.
3,"['리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 ...",['./data\\리비안_KR.txt'],[1],리비안의 초기 모델 R1의 원래 이름은 무엇인가요?,리비안의 초기 모델 R1의 원래 이름은 Avera입니다.
4,"['리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 ...",['./data\\리비안_KR.txt'],[1],리비안 R1의 디자인을 맡은 디자이너는 누구인가요?,리비안 R1의 디자인은 피터 스티븐스가 맡았습니다.
5,"['리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 ...",['./data\\리비안_KR.txt'],[1],리비안 R1은 어떤 구조적 특징을 가지고 있나요?,리비안 R1은 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 합니다.
6,"['리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱...",['./data\\리비안_KR.txt'],[2],리비안은 어떤 버전을 고려했나요?,"리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버..."
7,"['리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱...",['./data\\리비안_KR.txt'],[2],리비안의 첫 번째 양산 차량은 무엇인가요?,리비안의 첫 번째 양산 차량은 R1T 트럭입니다.
8,"['리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱...",['./data\\리비안_KR.txt'],[2],리비안은 언제 첫 번째 양산 차량을 고객에게 인도하기 시작했나요?,리비안은 2021년 10월에 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 ...
9,"['테슬라(Tesla, Inc\n)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 ...",['./data\\테슬라_KR.txt'],[3],테슬라는 어디에 본사를 두고 있나요?,테슬라는 텍사스주 오스틴에 본사를 두고 있습니다.


# 정보검색(Information Retrieval) 평가지표

<img src="information_Retrieval.png" width="800" align="left" />

## 테스트 데이터 만들기

RAG 시스템을 평가할 때 사용하는 '정답 문서', '검색 결과'의 예시 데이터셋을 만든다.

정답 데이터

In [150]:
actual_docs = [
    # query1: id가 1인 문서 하나만 찾으면 정답
    [
        Document(metadata={'id': 1}, page_content='Doc_1')
    ],
    # query2: id가 2와 5인 문서 두 개를 모두 찾아야 정답
    [
        Document(metadata={'id': 2}, page_content='Doc_2'),
        Document(metadata={'id': 5}, page_content='Doc_5')
    ],
]

검색 결과 데이터 - AI가 검색해온 문서

In [151]:
predicted_docs = [
    # query1: 2개 검색됨
    # query1는 2개를 가져왔고, 그 중 1위에 정답(id: 1)이 있으므로 검색 성능이 아주 좋다.
    [
        Document(metadata={'id': 1}, page_content='Doc_1'), # 첫 번째로 정답을 찾음
        Document(metadata={'id': 5}, page_content='Doc_5'), # 관련 없는 문서가 섞임
    ],
    # query2: 5개 검색됨
    # query2는 5개를 가져왔고, 정답인 id가 2와 5인 데이터가 뒤쪽(3, 4위)에 밀려나 있다. 이는 검색 알고리즘의 정확도가 다소 낮음을 의미한다.
    [
        Document(metadata={'id': 4}, page_content='Doc_4'), # 오답
        Document(metadata={'id': 1}, page_content='Doc_1'), # 오답
        Document(metadata={'id': 5}, page_content='Doc_5'), # 정답(3순위)
        Document(metadata={'id': 2}, page_content='Doc_2'), # 정답(4순위)
        Document(metadata={'id': 3}, page_content='Doc_3'), # 오답
    ],
]

OfflineRetrievalEvaluators는 RAG 시스템이나 정보 검색(Information Retrieval) 모델에서 검색기의 성능을 실시간 운영 전(오프라인 상태)에 정량적으로 평가하기 위해 사용되는 라이브러리를 사용하기 위해 `pip install krag`를 실행해서 설치하고 OfflineRetrievalEvaluators를 import 한다.

In [152]:
# !pip install krag

In [153]:
from krag.evaluators import OfflineRetrievalEvaluators

정답 데이터와 검색 결과 데이터를 비교할 평가 도구 객체를 생성한다.

In [154]:
evaluator = OfflineRetrievalEvaluators(
    # actual_docs(정답지): 사용자의 질문에 대해 '진짜 정답 문서' 리스트
    # 정답지 리스트의 형태는 [[정답1], [정답2, 정답3], ...]
    actual_docs=actual_docs,
    # predicted_docs(예측지): 검색기가 질문을 보고 검색기에서 '실제로 찾아온 문서' 리스트
    # 예측지 리스트의 형태는 [[결과1], [결과2, 결과3], ...]
    predicted_docs=predicted_docs
)

## 적중률(Hit Rate) 평가지표

적중률  
&nbsp;&nbsp;&nbsp;▶ 검색 결과에 관련 문서가 하나라도 포함되어 있는 비율로 검색 순서는 중요하지 않다.  
&nbsp;&nbsp;&nbsp;▶ 시스템이 관련 문서를 찾을 수 있는 능력을 나타낸다.  
&nbsp;&nbsp;&nbsp;▶ 범위는 0 ~ 1 사이이고 높을수록 좋다.

각 쿼리에 대한 검색 문서 중에서 실제 정답 문서가 포함되어 있으면 1, 그렇치 않으면 0으로 해서 전체 검색 쿼리에 대해서 평가를 진행하고 평균을 계산한다.

In [155]:
# k가 1일 경우 계산 방법 예시
# (첫 번째 쿼리 적중 여부 + 두 번째 쿼리 적중 여부) / 쿼리 개수
(1 + 0) / 2

0.5

In [156]:
# k가 2일 경우 계산 방법 예시
(1 + 0) / 2

0.5

In [157]:
# k가 3일 경우 계산 방법 예시 => 2번째 쿼리는 3개의 문서를 검색했을 때 정답이 1개만 포함되어 있으므로 못찾은 것으로 취급된다.
(1 + 0) / 2

0.5

In [158]:
# k가 4일 경우 계산 방법 예시 => 2번째 쿼리는 4개의 문서를 검색했을 때 정답이 2개 모두 포함되어 있으므로 찾은 것으로 취급된다.
(1 + 1) / 2

1.0

검색기가 찾아온 상위 k개의 결과에 실제 정답이 포함되어 있는지 확인하는 적중률을 측정한다.

In [159]:
k_values = [1, 2, 3, 4, 5]

for k in k_values:
    # calculate_hit_rate() 메소드의 인수로 검색해올 문서의 개수(k)를 넘겨 적중률을 측정한다.
    hit_rate = evaluator.calculate_hit_rate(k=k)
    # print(hit_rate)
    print(f'hit rate {k}: {hit_rate["hit_rate"]:.3f}')

hit rate 1: 0.500
hit rate 2: 0.500
hit rate 3: 0.500
hit rate 4: 1.000
hit rate 5: 1.000


결과가 이렇게 나오는 이유는 두 개의 질문 중 query1은 정답을 1등으로 찾았지만, query2는 정답을 3, 4등에 위치시켰기 때문이다.  
적중률은 상위 k개 안에 정답이 하나라도 포함되면 1점(성공), 없으면 0점(실패)으로 계산하여 평균을 낸다.

query1: 검색 결과 1위에 정답이 있다. 이미 k=1일 때 성공했다.  
query2: 검색 결과가 4위가 돼야 정답이 모두 있다. k=4가 되어서야 처음으로 성공한다.

k값에 따른 계산 과정  
hit rate 1, 2, 3은 query1은 1등에 정답이 있으니 성공(1점)이고 query2는 정답이 4등이 되어야 나오므로, 3등까지는 훑어보는 구간이고 실패(0정)이다.  
`(1 + 0) / 2 = 0.500`  
hit rate 4는 검색 범위를 4등까지 넓혔을 때 query2의 정답이 범위안에 들어오므로 성공(1점)으로 바뀐다.  
`(1 + 1) / 2 = 1.000`   
hit rate 5는 두 쿼리 모두 정답을 찾은 상태이므로 점수는 변하지 않는다.  
`(1 + 1) / 2 = 1.000`  

## 평균 역순위(Mean Reciprocal Rank, MRR) 평가지표

평균 역순위  
&nbsp;&nbsp;&nbsp;▶ 첫 번째 관련 문서의 역순위(`1 / 첫 번째 관련 문서의 순위`)의 평균으로 검색 순서를 고려한다.  
&nbsp;&nbsp;&nbsp;▶ 시스템이 관련 문서를 상위에 랭크시키는 능력을 나타낸다.  
&nbsp;&nbsp;&nbsp;▶ 범위는 0 ~ 1 사이이고 높을수록 좋다.

각 쿼리에 대해서 관련 문서가 처음 반환된 순위의 역수를 계산한 후 평균을 구하는 방법으로 사용자가 원하는 결과를 얼마나 빨리 찾을 수 있지를 평가한다.

In [160]:
# k가 1일 경우 계산 방법 예시
# (1 / 첫 번째 쿼리의 순위 + 1 / 두 번째 쿼리의 순위) / 쿼리 개수
(1 / 1 + 0 / 1) / 2

0.5

In [161]:
# k가 2일 경우 계산 방법 예시
(1 / 1 + 0 / 2) / 2

0.5

In [162]:
# k가 3일 경우 계산 방법 예시 => 2번째 쿼리는 3개의 문서가 관련 문서이다. 첫 번째 정답만 고려한다.
(1 / 1 + 1 / 3) / 2

0.6666666666666666

검색기가 찾아온 상위 k개의 결과에 실제 정답이 포함되어 있는지 확인하는 평균 역순위를 측정한다.

In [163]:
for k in k_values:
    # calculate_mrr() 메소드의 인수로 검색해올 문서의 개수(k)를 넘겨 적중률을 측정한다.
    mrr = evaluator.calculate_mrr(k=k)
    # print(mrr)
    print(f'mmr {k}: {mrr["mrr"]:.3f}')

mmr 1: 0.500
mmr 2: 0.500
mmr 3: 0.667
mmr 4: 0.667
mmr 5: 0.667


평균 역순위는 정답이 검색 결과의 몇 번째 순위에 처음 나타났는지를 수치화하며, 공식은 `1 / 순위` 이다.

query1은 정답이 1등으로 검색되므로 점수는 1 / 1 = 1점 이다.  
query2는 정답이 3등으로 처음 검색되므로 점수는 1 / 3 = 0.333점 이다.  

k값에 따른 계산 과정  
mmr 1은 query1은 1등이 정답이므로 1.000점이고 query2는 3등이 정답인데 1등까지만 보므로 0.000이 된다.  
`(1 + 0) / 2 = 0.500`  
mmr 2은 query1 여전히 1점이고 query2는 3등이 정답인데 2등까지만 보므로 0.000이 된다.  
`(1 + 0) / 2 = 0.500`  
mmr 3은 query1은 여전히 1점이고 query2는 3등이 정답인데 3등까지 보므로 0.333이 된다.  
`(1 + 0.333) / 2 = 0.667`  
mmr 4, 5는 query1은 여전히 1점이고 query2는 범위를 4, 5 등까지 넓혀도 3등에서 정답을 찾았으므로 그대로 유지된다.

## 정밀도(Percision) & 재현율(Recall) 평가지표

정밀도  
&nbsp;&nbsp;&nbsp;▶ 상위 k개의 검색 결과 중에서 관련 문서의 비율로 `상위 k개의 결과 중 관련 문서 개수 / k` 이다.  
&nbsp;&nbsp;&nbsp;▶ 검색 결과의 정확도를 나타낸다.  
&nbsp;&nbsp;&nbsp;▶ 범위는 0 ~ 1 사이이고 높을수록 좋다.

재현율  
&nbsp;&nbsp;&nbsp;▶ 상위 k개의 검색 결과에서 찾은 관련 문서의 비율로 `상위 k개의 결과 중 관련 문서 개수 / 전체 관련 문서 개수` 이다.  
&nbsp;&nbsp;&nbsp;▶ 시스템이 모든 관련 문서를 찾을 수 있는 능력을 나타낸다.  
&nbsp;&nbsp;&nbsp;▶ 범위는 0 ~ 1 사이이고 높을수록 좋다.

검색기가 찾아온 상위 k개의 결과에 실제 정답이 포함되어 있는지 확인하는 정밀도를 측정한다.

In [164]:
for k in k_values:
    # calculate_precision() 메소드의 인수로 검색해올 문서의 개수(k)를 넘겨 정밀도를 측정한다.
    percision = evaluator.calculate_precision(k=k)
    # print(percision)
    print(f'micro_precision {k}: {percision["micro_precision"]:.3f}, macro_precision {k}: {percision["macro_precision"]:.3f}')

micro_precision 1: 0.500, macro_precision 1: 0.500
micro_precision 2: 0.250, macro_precision 2: 0.250
micro_precision 3: 0.400, macro_precision 3: 0.417
micro_precision 4: 0.500, macro_precision 4: 0.500
micro_precision 5: 0.429, macro_precision 5: 0.450


k=3일 경우 쿼리별 정밀도 계산

query1은 검색 결과가 2개(id: 1, id: 5)로 3개를 채우지 못하고 2개만 반환해서 그 중에 실제 정답이 1개(id: 1)이므로 1 / 2 = 0.5점이 된다.  
query2는 검색 결과가 5개지만 상위 3개(id: 4, id: 1, id: 5)만 고려해서 그 중에 실제 정답이 1개(id: 5)이므로 1 / 3 = 0.333점이 된다.

최종 지표 산출

micro_precision은 전체 검색된 문서 수 대비 전체 적중 수로 계산한다.  
전체 검색된 문서 수는 query1은 2개이고 query2는 3개이므로 5개이다.  
전체 적중된 문서 수는 query1은 1개이고 query2도 1개이므로 2개이다.  
`2 / 5 = 0.4`  
macro_precision은 각 쿼리 점수의 단순 평균이다.  
`(0.5 + 0.333) = 0.417`

검색기가 찾아온 상위 k개의 결과에 실제 정답이 포함되어 있는지 확인하는 재현율를 측정한다.

In [165]:
for k in k_values:
    # calculate_recall() 메소드의 인수로 검색해올 문서의 개수(k)를 넘겨 재현율을 측정한다.
    recall = evaluator.calculate_recall(k=k)
    # print(recall)
    print(f'micro_recall {k}: {recall["micro_recall"]:.3f}, macro_recall {k}: {recall["macro_recall"]:.3f}')

micro_recall 1: 0.333, macro_recall 1: 0.500
micro_recall 2: 0.333, macro_recall 2: 0.500
micro_recall 3: 0.667, macro_recall 3: 0.750
micro_recall 4: 1.000, macro_recall 4: 1.000
micro_recall 5: 1.000, macro_recall 5: 1.000


k=3일 경우 쿼리별 재현율 계산

query1은 찾아야 할 정답(id: 1)이 1개이고 상위 3의 결과(id: 1, id: 5)는 이므로 정답 1개 중에서 1개를 찾아서 1점이 된다.  
query2는 찾아야 할 정답(id: 2, id: 5)이고 상위 3개의 결과(id: 4, id: 1, id: 5)는 이므로 정답 2개 중에서 1개만 찾아서 0.5점이 된다.

최종 지표 산출

micro_recall은 모든 정답 개수 대비 모든 적중 개수로 계산하며 개별 문서 하나하나의 적중 여부를 더 중요하게 볼 때 사용한다.  
전체 정답 개수는 query1은 1개이고 query2는 2개 이므로 3개이다.  
전체 적중 개수는 query1은 1개이고 query2는 1개 이므로 2개이다.  
`2 / 3 = 0.667`  
macro_recall은 각 쿼리 점수의 단순 평균이며 모든 질문을 동일하게 중요하게 취급할 때 사용한다.  
`(1 + 0.5) / 2 = 0.75`

## 평균 평균 정밀도(mean Average Percision, mAP)  평가지표

평균 평균 정밀도  
&nbsp;&nbsp;&nbsp;▶ 각 관련 문서를 검색할 때마다의 정확도의 평균으로 각 관련 문서 검색 시검에 정밀도 평균을 모든 쿼리의 평균을 계산한다.  
&nbsp;&nbsp;&nbsp;▶ 검색 시스템의 전반적인 성능을 나타낸다.  
&nbsp;&nbsp;&nbsp;▶ 범위는 0 ~ 1 사이이고 높을수록 좋다.

검색기가 찾아온 상위 k개의 결과에 실제 정답이 포함되어 있는지 확인하는 평균 평균 정밀도를 측정한다.

In [166]:
for k in k_values:
    # calculate_map() 메소드의 인수로 검색해올 문서의 개수(k)를 넘겨 평균 평균 정밀도를 측정한다.
    map_score = evaluator.calculate_map(k=k)
    # print(map_score)
    print(f'map {k}: {map_score["map"]:.3f}')

map 1: 0.500
map 2: 0.500
map 3: 0.583
map 4: 0.708
map 5: 0.708


평균 평균 정밀도를 계산하기 위해서는 먼저 각 질문별 AP(Average Percision)를 구해야 한다. AP는 정답을 하나씩 찾을 때만의 정밀도를 모두 더한뒤, 전체 정답의 개수로 나눈 값이다.

query1의 정답은 1개(id: 1)이고 query2의 정답은 2개(id: 2, id: 5)이다.

단계별 평균 평균 정밀도 계산 과정  

mAP 1  
&nbsp;&nbsp;&nbsp;▶ query1은 1위가 정답(id: 1)이고 정밀도는 1 / 1 = 1이다.(AP = 1 / 1 = 1)  
&nbsp;&nbsp;&nbsp;▶ query2는 1위가 오답(id: 4)이고 정밀도는 0 / 1 = 0이다.  
&nbsp;&nbsp;&nbsp;▶ `(1 + 0) / 2 = 0.5`  
mAP 2  
&nbsp;&nbsp;&nbsp;▶ query1은 2위가 정답(id: 5)이므로 추가 정답이 없다.(AP = 1)  
&nbsp;&nbsp;&nbsp;▶ query2는 2위가 오답(id: 1)이고 정밀도는 0 / 1 = 0이다.  
&nbsp;&nbsp;&nbsp;▶ `(1 + 0) / 2 = 0.5`  
mAP 3  
&nbsp;&nbsp;&nbsp;▶ query1은 변화없다.(AP = 1)  
&nbsp;&nbsp;&nbsp;▶ query2는 3위가 정답(id: 5)이고 정밀도는 1 / 3 = 0.333이다.(AP = 0.333 / 2 = 0.167)이다.  
&nbsp;&nbsp;&nbsp;▶ `(1 + 0.166) / 2 = 0.583`  
mAP 4  
&nbsp;&nbsp;&nbsp;▶ query1은 변화없다.(AP = 1)  
&nbsp;&nbsp;&nbsp;▶ query2는 4위가 정답(id: 2)이고 정밀도는 2 / 4 = 0.5이다.(AP = (0.333 + 0.5) / 2 = 0.416)이다.  
&nbsp;&nbsp;&nbsp;▶ `(1 + 0.416) / 2 = 0.708`  
mAP 5  
&nbsp;&nbsp;&nbsp;▶ query1은 변화없다.(AP = 1)  
&nbsp;&nbsp;&nbsp;▶ query2는 5위가 오답(id: 3)이므로 변화없다.  
&nbsp;&nbsp;&nbsp;▶ `(1 + 0.416) / 2 = 0.708`  

## 정규화된 감쇄 누적 이득(Normalized Discounted Cumulative Gain, NDCG) 평가지표

정규화된 감쇄 누적 이득  
&nbsp;&nbsp;&nbsp;▶ 검색 결과의 순위를 고려한 누적 이득(gain)의 정규화된 값으로 실제 DCG를 이상적인 DCG로 나눈 값이다.  
&nbsp;&nbsp;&nbsp;▶ 검색 결과의 순위와 관련성을 모두 고려한 성능 지표이다.  
&nbsp;&nbsp;&nbsp;▶ 범위는 0 ~ 1 사이이고 높을수록 좋다.

검색기가 찾아온 상위 k개의 결과에 실제 정답이 포함되어 있는지 확인하는 정규화된 감쇄 누적 이득을 측정한다.

In [167]:
for k in k_values:
    # calculate_map() 메소드의 인수로 검색해올 문서의 개수(k)를 넘겨 평균 평균 정밀도를 측정한다.
    ndcg = evaluator.calculate_ndcg(k=k)
    # print(ndcg)
    print(f'ndcg {k}: {ndcg["ndcg"]:.3f}')

ndcg 1: 0.500
ndcg 2: 0.500
ndcg 3: 0.653
ndcg 4: 0.785
ndcg 5: 0.785


정규화된 감쇄 누적 이득는 정답이 상단(1위)에 있을수록 높은 점수를 주며, IDCG(이상적인 점수)로 실제 점수를 나눠서 표준화한 지표이다.

IDCG(이상적인 점수)  
&nbsp;&nbsp;&nbsp;▶ query1은 정답이 1개이므로 1위에 정답이 있을 때가 최상이다.(IDCG = 1 / log<sub>2</sub>2 = 1)  
&nbsp;&nbsp;&nbsp;▶ query2는 정답이 2개이므로 1위와 2위에 정답이 있을 때가 최상이다.(IDCG = 1 / log<sub>2</sub>2 + 1 / log<sub>2</sub>3 = 1 + 0.631 = 1.631)  

단계별 정규화된 감쇄 누적 이득 계산 과정

ndcg 1  
&nbsp;&nbsp;&nbsp;▶ query1은 1위가 정답(id: 1)이고 점수는 1 / log<sub>2</sub>2 = 1.0 이다.(NDCG = 1 / 1 = 1)  
&nbsp;&nbsp;&nbsp;▶ query2는 1위가 오답(id: 4)이고 점수는 0이다.(NDCG = 0 / 1 = 0)  
&nbsp;&nbsp;&nbsp;▶ 평균 NDCG = `(1 + 0) / 2 = 0.5`  
ndcg 2  
&nbsp;&nbsp;&nbsp;▶ query1은 2위가 오답(id: 5)이고 점수는 변화없다.(NDCG = 1)  
&nbsp;&nbsp;&nbsp;▶ query2는 2위가 오답(id: 1)이고 점수는 0이다.(NDCG = 0)  
&nbsp;&nbsp;&nbsp;▶ 평균 NDCG = `(1 + 0) / 2 = 0.5`  
ndcg 3  
&nbsp;&nbsp;&nbsp;▶ query1은 3위가 없으므로 점수는 변화없다.(NDCG = 1)  
&nbsp;&nbsp;&nbsp;▶ query2는 3위가 정답(id: 5)이고 점수는 1 / log<sub>2</sub>4 = 0.5이다.(NDCG = 0.5 / 1.631 = 0.3065)  
&nbsp;&nbsp;&nbsp;▶ 평균 NDCG = `(1 + 0.3065) / 2 = 0.653`  
ndcg 4  
&nbsp;&nbsp;&nbsp;▶ query1은 4위가 없으므로 점수는 변화없다.(NDCG = 1)  
&nbsp;&nbsp;&nbsp;▶ query2는 4위가 정답(id: 2)이고 점수는 1 / log<sub>2</sub>5 = 0.4307이다.
&nbsp;&nbsp;&nbsp;▶ 누적 점수는 3위 정답(0.5) + 4위 정답(0.4307) = 0.9307이다. (NDCG = 0.9307 / 1.631 = 0.5706)  
&nbsp;&nbsp;&nbsp;▶ 평균 NDCG = `(1 + 0.5706) / 2 = 0.785`  
ndcg 5  
&nbsp;&nbsp;&nbsp;▶ query1은 5위가 없으므로 점수는 변화없다.(NDCG = 1)  
&nbsp;&nbsp;&nbsp;▶ query2은 5위가 오답이므로 추가 점수가 없다.(NDCG = 0.5706)  
&nbsp;&nbsp;&nbsp;▶ 평균 NDCG = `(1 + 0.5706) / 2 = 0.785`  

## 테스트 데이터로 평가하기 위해 데이터를 준비한다.

In [168]:
df_test_qa.head(3)

,context,source,doc_id,question,answer
0,['리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전...,['./data\\리비안_KR.txt'],[0],리비안은 언제 설립되었나요?,리비안은 2009년에 설립되었습니다.
1,['리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전...,['./data\\리비안_KR.txt'],[0],리비안의 본사는 어디에 있나요?,리비안의 본사는 미시간주 리보니아에 있습니다.
2,['리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전...,['./data\\리비안_KR.txt'],[0],리비안은 어떤 종류의 차량에 집중하고 있나요?,리비안은 자율 전기차에 집중하고 있습니다.


판다스 데이터프레임에 문자열 형태로 저장된 데이터를 Document 객체로 만든다.

데이터프레임과 인덱스를 인수로 받아서 데이터프레임의 각 행을 Document 객체로 만들고 Document 객체를 리스트에 저장해서 리턴하는 함수

In [181]:
# df_test_qa의 데이터 타입은 판다스 데이터프레임, idx의 데이터 타입은 정수형, 이 함수가 리턴하는 데이터 타입은 List[Document] 이다.
def context_to_document(df_test_qa: pd.DataFrame, idx: int) -> List[Document]:
    # idx로 넘어온 행의 'context', 'source', 'doc_id' 열의 데이터를 얻어온다.
    # 얻어오는 데이터가 ['내용1', '내용2', ...]와 같은 형태의 문자열일 경우 eval() 함수를 사용해서 파이썬 리스트로 변환해야 한다.
    context = eval(df_test_qa.context[idx])
    source = eval(df_test_qa.source[idx])
    doc_id = eval(df_test_qa.doc_id[idx])
    # print('context_to_document', type(context), type(source), type(doc_id))
    
    # 데이터프레임에서 얻어온 데이터로 생성할 Document 객체를 저장할 빈 리스트를 만든다.
    context_docs = []
    # zip() 함수로 context, source, doc_id를 묶어서 반복하며 Document 객체를 만들어서 리스트에 추가한다.
    for c, s, d in zip(context, source, doc_id):
        doc = Document(page_content=c, metadata={'source': s, 'doc_id': d})
        # print(doc)
        context_docs.append(doc)
    return context_docs

In [182]:
# 데이터프레임의 0번 행의 데이터를 Document 객체로 만든다.
print('0번 행의 context 데이터')
print(df_test_qa.context[0]) # print(df_test_qa.context.iloc[0])
print('-' * 100 + '\n')

context_docs = context_to_document(df_test_qa, 0)
context_docs

0번 행의 context 데이터
['리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다\n2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다\n주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)']
----------------------------------------------------------------------------------------------------



[Document(metadata={'source': './data\\리비안_KR.txt', 'doc_id': 0}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다\n2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다\n주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)')]

## BM25 검색기 + Kiwi 토크나이저

krag 라이브러리를 활용해서 한국어 형태소 분석 기반의 검색기를 만든다.

Kiwi 형태소 분석기를 활용하는 토크나이저를 사용하기 위해 KiwiTokenizer를 import 한다.  
문장을 단어 단위로 나눌 때, 한국어의 특성(조사, 어미 등)을 고려하여 정확하게 분리해주는 도구이다.

In [184]:
from krag.tokenizers import KiwiTokenizer

BM25 알고리즘을 사용하면서, 검색 결과와 함께 유사도 점수를 반환하는 검색기를 사용하기 위해 KiWiBM25RetrieverWithScore를 import 한다.  
질문과 가장 유사한 문서를 찾고, 그 문서가 얼마나 관련 있는지 숫자로 알려주는 도구이다.

In [185]:
from krag.retrievers import KiWiBM25RetrieverWithScore

KiwiTokenizer로 지정하는 토크나이저의 세부 옵션

`model_type`: 형태소 분석에 사용할 언어 모델을 지정한다.  
&nbsp;&nbsp;&nbsp;▶ `knlm(Kneser-Ney Language Model)`은 krag에서 가장 권장되는 모델이다.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;통계 기반의 Kneser-Ney Smoothing 기법을 사용해서 특정 단어 뒤에 어떤 형태소가 나올 확률이 높은 계산해서 문맥을 파악한다.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;단어의 모호성을 해소하는 능력이 탁월하다.('배'가 먹는 배인지, 타는 배인지, 내 배인지 문맥으로 판단.)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;긴 단어를 적절한 의미로 잘 나눠준다.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;RAG 시스템 구축 시 가장 적합하다.  
&nbsp;&nbsp;&nbsp;▶ `sbg(Skip-Gram)`은 비교적 가벼운 경량 모델로 단어 간의 상관관계를 벡터 공간의 거리로 계산하는 방식(Word2Vec)의 모델이다.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;knlm 모델보다 분석 속도가 빠르고 메모리를 적게 차지한다.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;아주 복잡한 문서가 아니라면 준수한 성능을 보여준다.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;실시간 채팅 상담이나 저사양 서버 환경 등 속도가 최우선일 때 사용한다.  
&nbsp;&nbsp;&nbsp;▶ `None`을 사용해서 모델을 지정하지 않으면 시스템 기본값(주로 knlm)이 적용된다. 코드의 명확성을 위해 직접 명시하는 것이 좋다.

`typos`: 사용자가 입력한 문장에 포함된 오타를 인식하고, 이를 올바른 형태소로 교정하여 분석한다.  
&nbsp;&nbsp;&nbsp;▶ `'basic'`은 가장 많이 사용되는 설정으로, 일반적인 수준의 오타를 처리한다.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;입력된 단어가 사전에 없더라고, 한두 글자의 차이를 계산해서 가장 확률이 높은 표준 단어로 매칭한다.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;'안녕하세용' => '안녕하세요', '앉녕하세요' => '안녕하세요' 형태로 매칭한다.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;일반적인 챗봇, QnA 시스템, 웹 검색 등.  
&nbsp;&nbsp;&nbsp;▶ `None`은 오타 교정 기능을 사용하지 않는다.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;오타를 교정하는 계산 과정이 없으므로 형태소 분석 속도가 빠르다.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;오타가 발생되면 해당 단어를 '미등록어'로 취급하거나 엉뚱하게 분리할 수 있다.  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;데이터 정제가 완벽히 끝난 뉴스 기사나 논문을 인덱싱할 때, 또는 매우 빠른 속도가 필요한 환경 등.

KiWiBM25 검색기를 만든다.

In [189]:
retriever_bm25_kiwi = KiWiBM25RetrieverWithScore(
    # 검색 대상이 될 전체 문서를 넣어준다. 검색기는 이 문서들을 미리 훑어보고(인덱싱), 나중에 질문이 들어오면 여기서 정답 후보를 찾는다.
    documents=final_docs,
    # KiwiTokenizer 클래스의 생성자로 형태소 분석에 사용할 언어모델, 오타 처리 방법을 넘겨서 검색 시 사용할 토크나이저의 세부 옵션을 설정한다.
    kiwi_tokenizer=KiwiTokenizer(
        # 형태소 분석에 사용할 언어 모델을 지정한다. 'knlm'는 RAG 시스템에서 권장되는 모델이다.
        model_type='knlm',
        # 일반적인 수준의 오타를 처리한다.
        typos='basic'
    ),
    # 검색 결과로 가져올 상위 문서의 개수를 지정한다.
    k=5
)

앞서 만든 KiWiBM25 검색기를 사용하여 실제로 질문을 던지고, 그 결과(관련 문서와 점수)를 확인한다.

In [196]:
# 테스트 데이터를 준비하고 및 확인한다.
question = df_test_qa.question[0]
print('질문\n', question, sep='')
print('-' * 100)
context = df_test_qa.context[0]
print('관련 문서\n', context, sep='')

질문
리비안은 언제 설립되었나요?
----------------------------------------------------------------------------------------------------
관련 문서
['리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다\n2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다\n주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)']


In [198]:
# invoke() 메소드의 인수로 질문을 넘겨서 KiWiBM25 검색기를 실행한다.
# 질문을 Kiwi 형태소 분석기로 나누고(오타 교정 포함) 나눠진 토큰들을 BM25 알고리즘에 대입하여 미리 인덱싱된 문서들과 비교해서 가장 유사도가 높은 상위 문서들을
# 리스트 형태로 반환한다.
retriever_docs = retriever_bm25_kiwi.invoke(question)
retriever_docs

[KragDocument(metadata={'source': './data\\리비안_KR.txt', 'doc_id': 0, 'bm25_score': 1.6756153958802293}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다\n2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다\n주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)'),
 KragDocument(metadata={'source': './data\\테슬라_KR.txt', 'doc_id': 3, 'bm25_score': 1.3330702210379728}, page_content='테슬라(Tesla, Inc\n)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다\n2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)'),
 KragDocument(metadata={'source': './data\\리비안_KR.txt', 'doc_id': 2, 'bm25_score': 1.095334401200554}, page_content='리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다\n2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다\n리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)'),
 KragDocument(metadata={'s

In [205]:
for doc in retriever_docs:
    print('유사도 점수:', doc.metadata['bm25_score'])
    print('검색된 문서\n', doc.page_content, sep='')
    print('-' * 100)

유사도 점수: 1.6756153958802293
검색된 문서
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다
2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다
주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
----------------------------------------------------------------------------------------------------
유사도 점수: 1.3330702210379728
검색된 문서
테슬라(Tesla, Inc
)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다
2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
----------------------------------------------------------------------------------------------------
유사도 점수: 1.095334401200554
검색된 문서
리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다
2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다
리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
-------------------------------------------------------------------------------------------------

RAG 시스템에서 검색 성능을 여러 검색 지표를 사용해서 계산하는 함수를 만든다.

LangChain 프레임워크에서 제공하느 모든 검색기(Retriever)의 최상위 부모 클래스를 데이터 타입으로 사용하기 위해 BaseRetriever를 import 한다.

In [206]:
from langchain_core.retrievers import BaseRetriever

인수로 질문과 답변이 저장된 판다스 데이터프레임과 검색기, 검색할 문서 개수를 넘겨받아 검색 성능을 계산하는 함수

In [267]:
def evaluate_qa_test(df_test_qa: pd.DataFrame, retriever: BaseRetriever, k: int = 2) -> dict:
    # 각 질문에 대한 실제 정답 문서들을 순서대로 담아둘 빈 리스트를 선언한다.
    context_docs = []
    # 검색기가 질문을 보고 실제로 찾아온 문서들을 담아둘 빈 리스트를 선언한다.
    retriever_docs = []
    df_test = df_test_qa.copy() # 작업할 사본
    
    for idx in range(len(df_test_qa)):
        # 현재 행 번호(idx)에 해당되는 질문을 가져온다.
        question = df_test.question[idx]
        # print(question)
        # 데이터프레임의 현재 행의 내용을 Document 객체로 만드는 context_to_document() 함수를 실행한다.
        context_doc = context_to_document(df_test, idx)
        # print(context_doc)
        # Document 객체를 실제 정답 문서들을 순서대로 담아둘 리스트에 추가한다.
        context_docs.append(context_doc)
        # 검색기에게 질문을 던져서 관련있는 문서들을 검색한다.
        retriever_doc = retriever.invoke(question)
        # print(len(retriever_doc))
        # print(retriever_doc)
        # 검색기가 찾아온 문서들을 실제로 찾아온 문서들을 담아둘 리스트에 추가하다.
        retriever_docs.append(retriever_doc)
        
    # 정답 리스트와 검색 결과 리스트를 비교하여 점수를 매겨주는 평가자 객체를 생성한다.
    evaluator = OfflineRetrievalEvaluators(actual_docs=context_docs, predicted_docs=retriever_docs)
    
    # 개별 평가 지표를 계산한다.
    # Hit Rate: 상위 k개 안에 정답 문서가 하나라도 포함되어 있으면 1, 없으면 0으로 계산한 평균이다.
    hit_rate = evaluator.calculate_hit_rate(k=k)['hit_rate']
    # MRR: 정답이 몇 번째 순위에 있느냐에 따라 점수를 차등 부여한다.(1위는 1점, 2위는 0.5점, ...)
    mrr = evaluator.calculate_mrr(k=k)['mrr']
    # mAP: 여러 개의 정답이 있을 때 검색 정확도의 평균이다.
    map_score = evaluator.calculate_map(k=k)['map']
    # NDCG: 검색 결과의 순서에 가중치를 두어 상위권에 정답이 올수록 높은 점수를 준다.
    ndcg = evaluator.calculate_ndcg(k=k)['ndcg']
    
    '''
    print(f'K = {k}')
    print('-' * 100)
    print(f'Hit Rate: {hit_rate}')
    print(f'MRR: {mrr}')
    print(f'mAP: {map_score}')
    print(f'NDCG: {ndcg}')
    print('-' * 100)
    '''
    
    # 개별 평가 지표를 계산 결과를 딕셔너리로 만들어 리턴한다.
    result = {
        'Hit Rate': hit_rate,
        'MRR': mrr,
        'mAP': map_score,
        'NDCG': ndcg
    }
    # return result
    return pd.Series(result)

In [268]:
# KiWiBM25RetrieverWithScore 클래스로 검색기를 선언할 때 지정했던 검색해올 문서 개수를 수정한다.
retriever_bm25_kiwi.k = 1
result = evaluate_qa_test(df_test_qa, retriever_bm25_kiwi, k=1)
print(result)

Hit Rate    0.888889
MRR         0.888889
mAP         0.888889
NDCG        0.888889
dtype: float64


In [269]:
retriever_bm25_kiwi.k = 2
result = evaluate_qa_test(df_test_qa, retriever_bm25_kiwi, k=2)
print(result)

Hit Rate    1.000000
MRR         0.944444
mAP         0.944444
NDCG        0.958992
dtype: float64


In [270]:
retriever_bm25_kiwi.k = 3
result = evaluate_qa_test(df_test_qa, retriever_bm25_kiwi, k=3)
print(result)

Hit Rate    1.000000
MRR         0.944444
mAP         0.944444
NDCG        0.958992
dtype: float64


## Chroma 벡터저장소 검색기

HuggingFace의 BAAI/bge-m3 임베딩 모델을 사용해서 텍스트 데이터를 벡터화 하고, Chroma 벡터저장소 저장해서 벡터검색기를 만든다.

In [271]:
# 임베딩 모델을 생성한다.
embeddings_model = HuggingFaceEmbeddings(model='BAAI/bge-m3')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [272]:
# Chroma 벡터저장소를 생성한다.
chroma_db = Chroma.from_documents(
    documents=final_docs, 
    embedding=embeddings_model,
    collection_name='hf_bge_m3',
    persist_directory='./chroma_db'
)

In [273]:
# Chroma 벡터검색기를 생성한다.
retriever_chroma_hf = chroma_db.as_retriever(search_kwargs={'k': 5})

In [274]:
result = evaluate_qa_test(df_test_qa, retriever_chroma_hf, k=1)
print(result)

Hit Rate    0.777778
MRR         0.777778
mAP         0.777778
NDCG        0.777778
dtype: float64


In [275]:
result = evaluate_qa_test(df_test_qa, retriever_chroma_hf, k=2)
print(result)

Hit Rate    1.000000
MRR         0.888889
mAP         0.888889
NDCG        0.917984
dtype: float64


In [276]:
result = evaluate_qa_test(df_test_qa, retriever_chroma_hf, k=3)
print(result)

Hit Rate    1.000000
MRR         0.888889
mAP         0.888889
NDCG        0.917984
dtype: float64


서로 다른 방식의 검색기들을 하나로 합쳐 장점만 취하는 앙상블 검색기를 만들어서 키워드 기반(BM25)과 의미 기반(Chroma) 검색을 섞어 사용한다.

In [277]:
# 앞에서 retriever_bm25_kiwi.k=3 문장이 실행되서 retriever_bm25_kiwi가 검색해오는 문서 개수가 3개이고 retriever_chroma_hf는 검색해오는 문서가 5개이므로
# 두 검색기의 보폭을 맞춘다.
retriever_bm25_kiwi.k = 5

# 하나로 합칠 검색기들을 리스트 형태로 묶어준다.
# retriever_bm25_kiwi는 정확한 단어 일치를 잘 찾는 키워드 검색기이고 retriever_chroma_hf는 의미(맥락)을 잘 파악하는 벡터검색기 이다.
ensemble_retrievers = [retriever_bm25_kiwi, retriever_chroma_hf]
# 앙상블 검색기를 만든다.
ensemble_retriever = EnsembleRetriever(
    # 앙상블 검색기에서 사용할 서로 다른 검색기를 지정한다.
    retrievers=ensemble_retrievers,
    # 서로 다른 검색기 검색 결과의 가중치를 설정한다.
    weights=[0.5, 0.5]
)

In [278]:
result = evaluate_qa_test(df_test_qa, ensemble_retriever, k=1)
print(result)

Hit Rate    0.944444
MRR         0.944444
mAP         0.944444
NDCG        0.944444
dtype: float64


In [279]:
result = evaluate_qa_test(df_test_qa, ensemble_retriever, k=2)
print(result)

Hit Rate    1.000000
MRR         0.972222
mAP         0.972222
NDCG        0.979496
dtype: float64


In [280]:
result = evaluate_qa_test(df_test_qa, ensemble_retriever, k=3)
print(result)

Hit Rate    1.000000
MRR         0.972222
mAP         0.972222
NDCG        0.979496
dtype: float64
